In [ ]:
######################################################### Endothelial产图的code   ######### 
import h5py
import numpy as np
import scanpy as sc
import scipy.sparse as sp_sparse
import os
import pandas as pd
import gzip
import time
import re
#from pyensembl import EnsemblRelease
import pwd
import itertools
#### scanpy parameters
import omicverse as ov
import sys
import pandas as pd
#sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
#sc.logging.print_header()
#sc.settings.set_figure_params(dpi=80, facecolor="white")
import scvelo as scv
import cellrank as cr
from anndata import AnnData
import matplotlib.pyplot as plt
import dynamo as dyn
from scipy.stats import entropy
from sklearn.neighbors import NearestNeighbors
import stream as st
import biolearn



In [ ]:
adata = sc.read_h5ad('external_inputs/Endo_human_EndoDEG_count_notAge_526080.h5ad')   #



In [ ]:
# sorted(dir())



In [ ]:
# adata = sc.read_h5ad('external_inputs/Endo_human_EndoDEG_count_notAge_585468.h5ad')
adata = sc.read_h5ad('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')
adata.shape


In [ ]:
cell_lineage_dict = {
    "T": [
        "CD3D", "CD3E", "CD3G",  # T细胞核心标记
        "CD4", "CD8A", "CD8B",    # CD4+/CD8+ T亚群
        "FOXP3", "IL2RA",          # Treg细胞标记
        "TRAC", "TRBC1", "TRBC2"  # TCR基因
    ],
    "B": [
        "CD19", "CD79A", "CD79B",  # B细胞核心标记
        "MS4A1",                   # CD20
        "IGHD", "IGHM",            # B细胞受体
        "CD27", "CD38"             # 记忆B/浆细胞标记
    ],
    "Fibroblasts": [
        "COL1A1", "COL1A2",        # 胶原蛋白
        "COL3A1", "COL6A1", 
        "DCN", "LUM",              # 基质蛋白
        "FAP", "PDPN",             # 成纤维活化标记
        "ACTA2"                    # 肌成纤维标记（α-SMA）
    ],
    "Endothelial": [
        "PECAM1", "CDH5",         # 内皮粘附分子
        "VWF", "CLDN5",           # Weibel-Palade小体标记
        "SELE", "ICAM1",           # 活化内皮标记
        "FLT1", "KDR"              # VEGF受体
    ],
    "Mono/Macro": [
        "CD14", "CD68",            # 单核/巨噬核心标记
        "FCGR3A", "CD16",          # 非经典单核
        "LYZ", "CST3",             # 溶酶体相关
        "S100A8", "S100A9",       # 钙结合蛋白
        "ITGAM"                    # CD11b
    ],
    "Epithelial": [
        "EPCAM", "CDH1",           # 上皮粘附分子
        "KRT8", "KRT18",           # 角蛋白
        "MUC1", "TFF3",            # 分泌蛋白
        "SFN", "CLDN3"             # 紧密连接蛋白
    ]
}


In [ ]:
####### Endo subtype marker dict
Endo_marker_dict = {
    "SMC": ["TAGLN","RG55","NDUFA4L2"],
    "PC": ["COL1A1", "COL3A1", "COL1A2"],
    "LEC": ["CCL21", "TFF3", "MMRN1"],
    "VenEC": ["ACKR1", "CLU", "CCL14"],
    "CapEC": ["FLT1", "PLVAP", "HSPG2"],
    "ArtEC": ["CXCL12", "IGFBP3", "SRGN"],
    "EndoMT-I": [
        "RGS5", "BGN", "NOTCH3", "NDUFA4L2"
    ],
    "EndoMT-II": [
        'APOD', 'CFD', 'DCN', 'CXCL14'
    ]
}


In [ ]:
adata.obs.columns


In [ ]:
gene_marker_list = [gene for gene in cell_lineage_dict['T'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='T')
gene_marker_list = [gene for gene in cell_lineage_dict['B'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='B')
gene_marker_list = [gene for gene in cell_lineage_dict['Fibroblasts'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Fibro')
gene_marker_list = [gene for gene in cell_lineage_dict['Endothelial'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Endo')
gene_marker_list = [gene for gene in cell_lineage_dict['Mono/Macro'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='MoMac')
gene_marker_list = [gene for gene in cell_lineage_dict['Epithelial'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Epi')   


In [ ]:
#### adata.obs.columns   
sc.pl.umap(adata, color=['score_antigen_pre',
       'score_Tip', 'score_stalk'], 
           color_map='viridis', 
           frameon=False,
           ncols=3,
           size=3,
               show=False,  # 不立即显示，方便后续调整
    legend_fontsize=12,  # 调整图例字体大小
    legend_fontoutline=2,  # 可选，图例字体轮廓
           wspace=0.1)

import matplotlib.pyplot as plt
plt.gcf().set_size_inches(18, 5)  # 调整图形大小
plt.tight_layout()  # 调整布局
plt.show()


In [ ]:
# 在降维图上展示signature score
sc.pl.umap(adata, color=['T','B','Fibro', 'Endo','MoMac', 'Epi'], 
           color_map='viridis', 
           frameon=False,
           ncols=3,
           size=3,
               show=False,  # 不立即显示，方便后续调整
    legend_fontsize=12,  # 调整图例字体大小
    legend_fontoutline=2,  # 可选，图例字体轮廓
           wspace=0.1)

import matplotlib.pyplot as plt
plt.gcf().set_size_inches(18, 10)  # 调整图形大小
plt.tight_layout()  # 调整布局
plt.show()


In [ ]:
# list(Endo_marker_dict.keys())


In [ ]:
# help(sc.tl.score_genes)


In [ ]:
# adata.raw.X.max()


In [ ]:
adata.obs.columns


In [ ]:
gene_marker_list = [gene for gene in Endo_marker_dict['SMC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='SMC')
gene_marker_list = [gene for gene in Endo_marker_dict['PC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='Peri')
gene_marker_list = [gene for gene in Endo_marker_dict['LEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='LEC')
gene_marker_list = [gene for gene in Endo_marker_dict['VenEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='VenEC')
gene_marker_list = [gene for gene in Endo_marker_dict['CapEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='CapEC')
gene_marker_list = [gene for gene in Endo_marker_dict['ArtEC'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='ArtEC')   
gene_marker_list = [gene for gene in Endo_marker_dict['EndoMT-I'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-I')
gene_marker_list = [gene for gene in Endo_marker_dict['EndoMT-II'] if gene in adata.var_names]
sc.tl.score_genes(adata, gene_marker_list, score_name='EndoMT-II') 


In [ ]:
adata.obs[['SMC', 'Peri', 'LEC', 'VenEC', 'CapEC', 'ArtEC',
       'EndoMT-I', 'EndoMT-II']]


In [ ]:
adata.raw.X[0:10,0:10].max()


In [ ]:
# 在降维图上展示signature score
sc.pl.umap(adata, color=['SMC','Peri',  'LEC', 'VenEC', 'CapEC', 'ArtEC', 'EndoMT-I', 'EndoMT-II'], 
           color_map='viridis', 
           frameon=False,
           ncols=4,
           size=3,use_raw=False,
               show=False,  # 不立即显示，方便后续调整
    legend_fontsize=12,  # 调整图例字体大小
    legend_fontoutline=2,  # 可选，图例字体轮廓
           wspace=0.1)

import matplotlib.pyplot as plt
plt.gcf().set_size_inches(24, 10)  # 调整图形大小
plt.tight_layout()  # 调整布局
plt.show()



In [ ]:
adata.obs[['leiden','Celltype','Celltype_general']].drop_duplicates().sort_values('leiden')


In [ ]:
# adata.X.min()

In [ ]:
# adata.X.max()


In [ ]:
adata.raw.X.max()


In [ ]:
############### Endothelial 之前 58万个细胞的时候
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
# 1. 数据准备与检查
# ----------------------
# 假设你的 AnnData 对象已经加载并命名为 adata
# adata = sc.read_h5ad("your_data.h5ad")
# 确保 UMAP 坐标已经计算好
if 'X_umap' not in adata.obsm:
    sc.tl.umap(adata)

# 检查 obs 列是否存在
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要的列: {col}")

# ----------------------
# 2. 定义颜色映射
# ----------------------
# 为每个 Celltype_general 定义一个主色调
# ['PC', 'VenEC', 'EndoMT-II', 'CapEC', 'LymEC', 'SMC', 'ArtEC', 'EndoMT-I']
general_colors = {
    'PC': '#1f77b4',       # 深海蓝（Cell经典主色）
    'VenEC': '#ff7f0e',    # 暖橙（Nature常用强调色）
    'EndoMT-II': '#2ca02c',# 森林绿（Science功能色）
    'CapEC': '#d62728',    # 猩红（顶级期刊高亮色）
    'LymEC': '#9467bd',    # 紫罗兰（Cell子刊常用）
    'SMC': '#8c564b',      # 棕褐（Nature子刊纹理色）
    'ArtEC': '#e377c2',    # 蔷薇粉（Science子刊对比色）
    'EndoMT-I': '#7f7f7f'  # 石墨灰（期刊中性对比色)
}

# 为每个具体的 Celltype 生成渐变颜色，使其在同一 general 组内色调统一
celltype_to_color = {}
for general_type in adata.obs['Celltype_general'].unique():
    # 获取该 general 组下的所有 celltype
    celltypes_in_group = adata.obs[adata.obs['Celltype_general'] == general_type]['Celltype'].unique()
    # 生成渐变颜色
    n = len(celltypes_in_group)
    if n == 0:
        continue
    # 从主色调生成渐变
    base_color = general_colors[general_type]
    # 使用 matplotlib 的 colormap 生成渐变
    cmap = plt.cm.get_cmap(plt.cm.colors.LinearSegmentedColormap.from_list("", [base_color, "#ffffff"]), n)
    for i, celltype in enumerate(celltypes_in_group):
        celltype_to_color[celltype] = cmap(i / (n - 1)) if n > 1 else base_color

# ----------------------
# 3. 绘制 UMAP 图（核心修改部分）
# ----------------------
plt.rcParams['figure.figsize'] = (12, 6)
fig, ax = plt.subplots()

# 用 leiden 聚类绘制 UMAP 散点图
sc.pl.umap(adata, color='leiden', ax=ax, show=False, legend_loc='on data',title='Endothelial Atlas', s=10, alpha=0.8)
# 设置标题字体：字号调大+加粗
ax.set_title('Endothelial Atlas', fontsize=16, fontweight='bold', pad=20)

# ----------------------
# 4. 移除图表边框
# 隐藏所有坐标轴边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
# 隐藏刻度（可选，根据文章风格调整）
ax.set_xticks([])
ax.set_yticks([])
# 关键：移除 scanpy 自带的 UMAP1/UMAP2 轴标签
ax.set_xlabel('')
ax.set_ylabel('')

# ----------------------
# 5. 添加带箭头的轴标签
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%

# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=12,
    ha='left',  # 文本对齐方式
    va='center'
)
# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=12,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# 6. 创建两个独立的图例（核心修改）
# ----------------------
# 第一组：qFibro + Myfibro + iFibro（左侧图例）
col1_generals = ['PC', 'VenEC', 'LymEC', 'SMC', 'EndoMT-I']
col1_handles = []
col1_labels = []
for general_type in col1_generals:
    # 添加 general_type 标题
    title_patch = mpatches.Patch(color=general_colors[general_type], label=general_type)
    col1_handles.append(title_patch)
    col1_labels.append(general_type)
    
    # 添加该组下所有 celltype 条目
    celltypes_in_group = sorted(adata.obs[adata.obs['Celltype_general'] == general_type]['Celltype'].unique())
    for celltype in celltypes_in_group:
        patch = mpatches.Patch(color=celltype_to_color[celltype], label=celltype)
        col1_handles.append(patch)
        col1_labels.append(f"  {celltype}")

# 创建第一个图例（位置：UMAP图右侧偏上）
legend1 = ax.legend(
    handles=col1_handles, 
    labels=col1_labels, 
    bbox_to_anchor=(1.02, 0.95),  # 右侧偏上
    loc='upper left', 
    borderaxespad=0, 
    frameon=False,
    fontsize=10  # 调整字体大小适配
)
ax.add_artist(legend1)  # 关键：保留第一个图例

# 第二组：eFibro（右侧图例，在第一个图例下方）
col2_generals = ['ArtEC','CapEC','EndoMT-II']
col2_handles = []
col2_labels = []
for general_type in col2_generals:
    # 添加 general_type 标题
    title_patch = mpatches.Patch(color=general_colors[general_type], label=general_type)
    col2_handles.append(title_patch)
    col2_labels.append(general_type)
    
    # 添加该组下所有 celltype 条目
    celltypes_in_group = sorted(adata.obs[adata.obs['Celltype_general'] == general_type]['Celltype'].unique())
    for celltype in celltypes_in_group:
        patch = mpatches.Patch(color=celltype_to_color[celltype], label=celltype)
        col2_handles.append(patch)
        col2_labels.append(f"  {celltype}")

# 创建第二个图例（位置：第一个图例下方）
legend2 = ax.legend(
    handles=col2_handles, 
    labels=col2_labels, 
    bbox_to_anchor=(1.3, 0.95),  # 右侧偏下，与第一个图例错开
    loc='upper left', 
    borderaxespad=0, 
    frameon=False,
    fontsize=10  # 统一字体大小
)

# 7. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height
        }
    return bounds

# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 8. 为每个Celltype_general添加弯曲虚线圆圈和文本标签（核心新增）
# ----------------------
for general_type, color in general_colors.items():
    if general_type not in cluster_bounds:
        continue
    # 获取该分组的边界信息
    bounds = cluster_bounds[general_type]
    center_x, center_y = bounds['center']
    width = bounds['width']
    height = bounds['height']   
    # 创建弯曲的椭圆虚线框（模拟弯曲圈）
    ellipse = mpatches.Ellipse(
        (center_x, center_y),
        width=width,
        height=height,
        edgecolor=color,
        facecolor='none',
        linestyle='--',  # 虚线
        linewidth=2,     # 线宽
        alpha=0.8        # 透明度
    )
    ax.add_patch(ellipse) 
    # 添加文本标签（在圈的右侧/上方，避免遮挡）
    # 计算标签位置（圈的右侧偏移一点）
    label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
    label_y = center_y  
    # 添加文本
    ax.text(
        label_x, label_y,
        general_type,
        fontsize=12,
        fontweight='bold',
        color=color,
        ha='left',
        va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
    )

# ----------------------
# 9. 保存图片为PDF格式
# ----------------------
# 调整布局，确保图例和内容完整
plt.tight_layout()
# 保存为PDF（bbox_inches='tight' 确保图例完整保存，dpi保证清晰度）
fig_dir='./TabulaStromal_figures/Endothelial/'
# plt.savefig(fig_dir+"Endothelial_umap_leiden_Celltype.pdf", dpi=300, bbox_inches='tight', format='pdf')
# 显示图片（可选）
plt.show()


In [ ]:
Nat_color_list = ['#E64B35',  # Nature红
        '#4DBBD5',  # Nature蓝
        '#00A087',  # Nature绿
        '#F39B7F',  # 珊瑚色
        '#8491B4',  # 淡紫色
        '#91D1C2',  # 薄荷绿
        '#DC0000',  # 正红
        '#7E6148',  # 棕色
        '#F4649D',  # 粉红
        '#7C6A9C',  # 紫罗兰
        '#AF7AC4',  # 浅紫
        '#FFBB78',  # 杏色
        '#98DF8A',  # 嫩绿
        '#F7B6D2',  # 浅粉
        '#C5B0D5',  # 淡紫
        '#FF9896',  # 浅红
        '#9467BD',  # 深紫
        '#8C564B',  # 赭石
        '#E377C2',  # 玫红

       '#BCBD22',  # 黄绿
        '#17BECF',  # 青绿
        '#AEC7E8',  # 天蓝
     
        '#9EDAE5',  # 浅青
      '#C2B0B0',  # 灰粉
        '#6B8E23',  # 橄榄绿
        '#FF8C00'  # 暗橙
    ]

len(Nat_color_list)


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
# 1. 基础检查
required_cols = ['leiden', 'Celltype_general', 'Celltype']
for col in required_cols:
    if col not in adata.obs.columns:
        raise ValueError(f"adata.obs 中缺少必要列: {col}")

# 确保 leiden 是字符串（Scanpy 推荐）
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
# 根据adata.obs['leiden'] 产生 leiden color list, 参考Nature系列的配色
# ================================
# 2. 构建 Leiden → 颜色映射
# ================================
leiden_categories = sorted(
    adata.obs['leiden'].unique(),
    key=lambda x: int(x)
)
if len(leiden_categories) != len(Nat_color_list):
    raise ValueError("Nat_color_list 数量必须等于 Leiden cluster 数量")

leiden_color_dict = dict(zip(leiden_categories, Nat_color_list))
# 告诉 scanpy 使用这套颜色
adata.uns['leiden_colors'] = [
    leiden_color_dict[c] for c in leiden_categories
]
# ================================
# 3. 构建 Celltype_C 列
# ================================

adata.obs['Celltype_C'] = (
    adata.obs['Celltype'] +
    "_C" +
    adata.obs['leiden']
)
# ================================
# 4. 构建 Celltype_C → 颜色映射（来自 Leiden）
# ================================
celltypeC_color_dict = {}
for leiden_id in leiden_categories:
    mask = adata.obs['leiden'] == leiden_id
    celltypeC_values = adata.obs.loc[mask, 'Celltype_C'].unique()
    for val in celltypeC_values:
        celltypeC_color_dict[val] = leiden_color_dict[leiden_id]

# ================================
# 5. 开始绘图
# ================================
plt.rcParams['figure.figsize'] = (13, 6)
fig, ax = plt.subplots()
# 关键：指定 palette=Nat_color_list
sc.pl.umap(
    adata,
    color='leiden',
    palette=Nat_color_list,
    ax=ax,
    show=False,
    legend_loc='on data',   # 不用scanpy默认legend
    s=10,
    alpha=0.8
)
ax.set_title('Endothelial Atlas', fontsize=16, fontweight='bold', pad=20)
# ================================
# 6. 去边框
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# ----------------------
# 5. 添加带箭头的轴标签
# ----------------------
# 计算箭头长度（基于UMAP坐标范围，确保箭头比例协调）
x_range = ax.get_xlim()
y_range = ax.get_ylim()
arrow_length_x = (x_range[1] - x_range[0]) * 0.1  # 占X轴范围10%
arrow_length_y = (y_range[1] - y_range[0]) * 0.1  # 占Y轴范围10%
# 添加UMAP_1箭头（x轴0点向右）
ax.annotate(
    'UMAP_1',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(arrow_length_x, 0),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='left',  # 文本对齐方式
    va='center'
)
# 添加UMAP_2箭头（y轴0点向上，竖直文本）
ax.annotate(
    'UMAP_2',
    xy=(0, 0),  # 箭头起点（坐标轴0点）
    xytext=(0, arrow_length_y),  # 箭头终点
    arrowprops=dict(
        arrowstyle='<-',  # 箭头样式
        color='black',    # 箭头颜色
        lw=1.5            # 箭头线宽
    ),
    fontsize=13,
    ha='center',  # 文本对齐方式
    va='bottom',
    rotation=90   # 文本竖直排布
)

# ================================
# 8. 构建分组 legend（按 Celltype_general）
# ================================
general_groups =  ['Peri', 'VenEC', 'LymEC', 'SMC', 'EndoMT-I','ArtEC','CapEC','EndoMT-II']
col1_generals = ['Peri', 'VenEC', 'LymEC', 'SMC', 'EndoMT-I']
col2_generals = ['ArtEC','CapEC','EndoMT-II']
def build_legend_handles(generals):
    handles = []
    labels = []
    for g in generals:
        subset = adata.obs[adata.obs['Celltype_general'] == g]
        unique_celltypeC = sorted(subset['Celltype_C'].unique())
        # 添加 general 标题（黑色）
        handles.append(mpatches.Patch(color='white', label=g))
        labels.append(g)
        # 添加每个 Celltype_C
        for ct in unique_celltypeC:
            handles.append(
                mpatches.Patch(
                    color=celltypeC_color_dict[ct],
                    label=ct
                )
            )
            labels.append("  " + ct)  
    return handles, labels


# 左 legend
handles1, labels1 = build_legend_handles(col1_generals)
legend1 = ax.legend(
    handles1,
    labels1,
    bbox_to_anchor=(1.02, 0.95),
    loc='upper left',
    frameon=False,
    fontsize=9
)
ax.add_artist(legend1)
# 右 legend
handles2, labels2 = build_legend_handles(col2_generals)
legend2 = ax.legend(
    handles2,
    labels2,
    bbox_to_anchor=(1.32, 0.95),
    loc='upper left',
    frameon=False,
    fontsize=9
)
# ----------------------
# 8. 计算每个 Celltype_general 对应的UMAP坐标边界（用于画圈）
# ----------------------
def get_cluster_bounds(adata, group_col):
    """计算每个分组的UMAP坐标边界（中心+宽高）"""
    bounds = {}
    for group in adata.obs[group_col].unique():
        # 获取该分组的UMAP坐标
        umap_coords = adata.obsm['X_umap'][adata.obs[group_col] == group]
        if len(umap_coords) == 0:
            continue
        # 计算中心坐标
        center_x = np.mean(umap_coords[:, 0])
        center_y = np.mean(umap_coords[:, 1])
        # 计算边界（用于确定圆圈大小）
        min_x, max_x = np.min(umap_coords[:, 0]), np.max(umap_coords[:, 0])
        min_y, max_y = np.min(umap_coords[:, 1]), np.max(umap_coords[:, 1])
        # 计算宽度和高度（扩大1.2倍，让圈更宽松）
        width = (max_x - min_x) * 1.2
        height = (max_y - min_y) * 1.2
        bounds[group] = {
            'center': (center_x, center_y),
            'width': width,
            'height': height
        }
    return bounds

# 计算每个Celltype_general的UMAP边界
cluster_bounds = get_cluster_bounds(adata, 'Celltype_general')

# ----------------------
# 9. 为每个Celltype_general添加弯曲虚线圆圈和文本标签
general_colors = {
    'Peri': '#1f77b4',       # 深海蓝（Cell经典主色）
    'VenEC': '#ff7f0e',    # 暖橙（Nature常用强调色）
    'EndoMT-II': '#2ca02c',# 森林绿（Science功能色）
    'CapEC': '#d62728',    # 猩红（顶级期刊高亮色）
    'LymEC': '#9467bd',    # 紫罗兰（Cell子刊常用）
    'SMC': '#8c564b',      # 棕褐（Nature子刊纹理色）
    'ArtEC': '#e377c2',    # 蔷薇粉（Science子刊对比色）
    'EndoMT-I': '#7f7f7f'  # 石墨灰（期刊中性对比色）
}

# ----------------------
for general_type, color in general_colors.items():
    if general_type not in cluster_bounds:
        continue
        
    # 获取该分组的边界信息
    bounds = cluster_bounds[general_type]
    center_x, center_y = bounds['center']
    width = bounds['width']
    height = bounds['height']
    # 创建弯曲的椭圆虚线框（模拟弯曲圈）
    ellipse = mpatches.Ellipse(
        (center_x, center_y),
        width=width,
        height=height,
        edgecolor=color,
        facecolor='none',
        linestyle='--',  # 虚线
        linewidth=2,     # 线宽
        alpha=0.8        # 透明度
    )
    ax.add_patch(ellipse)
    # 添加文本标签（在圈的右侧/上方，避免遮挡）
    # 计算标签位置（圈的右侧偏移一点）
    label_x = center_x + width/2 + (x_range[1]-x_range[0])*0.02
    label_y = center_y
    # 添加文本
    ax.text(
        label_x, label_y,
        general_type,
        fontsize=12,
        fontweight='bold',
        color=color,
        ha='left',
        va='center',
        bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=color, alpha=0.7)
    )

# 10. 保存
plt.tight_layout()
fig_dir='./TabulaStromal_figures/Endothelial/'
plt.savefig(
    fig_dir+"Endothelial_umap_leiden_Celltype.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf'
)
plt.show()


In [ ]:
len(adata.obs['leiden'].unique())


In [ ]:
# adata.obs[['Celltype','leiden']].drop_duplicates()


In [ ]:
adata.obs.columns


In [ ]:
celltypes_in_group

In [ ]:
adata_sub = adata[(adata.obs['Age_group']!='Unknown')]


In [ ]:
########### Endothelial中有的tissue的细胞数很少
check_tissue_endo = ['kidney','heart','none','bone marrow','airway']
meta_data =adata.obs
meta_data[meta_data['tissue'].isin(check_tissue_endo)][['GSE','GSM','sample']].drop_duplicates()


In [ ]:
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 300

sc.pl.umap(
    adata,
    color='Celltype_general',
    frameon=False,      # 去掉边框
    show=False, # 不立即显示，以便自定义
    # legend_loc='on data'
)


In [ ]:
plt.savefig('./TabulaStromal_figures/Endothelial/Endo_celltype_general_umap.pdf', bbox_inches='tight', transparent=True)


In [ ]:
adata.obs.columns


In [ ]:
column_mapping = {
    'cell_type_lvl1': 'Celltype_general',
    'cell_type_lvl2': 'Celltype_sub', 
    'cell_type_tissue': 'Celltype'
}

# 重命名列
adata.obs = adata.obs.rename(columns=column_mapping)


In [ ]:
meta_data = adata.obs



In [ ]:
adata.obs.columns


In [ ]:
### 
sorted(dir())


In [ ]:
meta_data.columns


In [ ]:
adata.obs['Age_group'].unique()


In [ ]:
Fibro_meta = pd.read_csv('',sep='\t')

In [ ]:
Fibro_meta


In [ ]:

# meta_data.to_csv('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t',index=False)
meta_data = pd.read_csv('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t')


In [ ]:
meta_data = meta_data[~(meta_data['Age_group']=='Unknown')]
meta_data 


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import numpy as np

def plot_box(meta_data, age_group, age_order, cell_subtype):
    # 计算细胞类型在每个个体中的比例
    cell_counts = meta_data.groupby(['sample', cell_subtype]).size().reset_index(name='count')
    total_counts = meta_data.groupby('sample').size().reset_index(name='total_count')
    # total_counts出现了一些adata中没有的个体，导致NaN,把他们去掉
    total_counts = total_counts[total_counts['sample'].isin(meta_data['sample'])]
    cell_counts = cell_counts.merge(total_counts, on='sample')
    cell_counts['percentage'] = cell_counts['count'] / cell_counts['total_count'] * 100  # 转换为百分比

    # 添加 Age_group（作为标识）
    cell_counts = cell_counts.merge(meta_data[['sample', age_group]].drop_duplicates(), on='sample', how='left')
    cell_counts['Age_group'] = pd.Categorical(cell_counts[age_group], categories=age_order, ordered=True)

    # 按年龄组排序（如果需要）
    cell_counts = cell_counts.sort_values(by='Age_group')
    # 作图
    cell_types = cell_counts[cell_subtype].unique()

    sns.set_style("whitegrid")

    for cell in cell_types:
        plt.figure(figsize=(8, 5))
        data_subset = cell_counts[cell_counts[cell_subtype] == cell]
        
        # 计算p值和效应量时排除A和H组
        filtered_data = data_subset[~data_subset['Age_group'].isin(['A', 'H'])]

        # Kruskal-Wallis H 检验，计算整体差异的 p 值
        groups = filtered_data['Age_group'].unique()
        group_data = [filtered_data[filtered_data['Age_group'] == group]['percentage'] for group in groups]
        # 计算 p 值
        p_val = stats.kruskal(*group_data)[1]
    
        # 计算 Eta Squared (η²) for Kruskal-Wallis
        n = sum([len(group) for group in group_data])  # 总样本量
        h_stat = stats.kruskal(*group_data)[0]  # Kruskal-Wallis H 统计量
        eta_squared = h_stat**2 / (h_stat**2 + n - len(groups))  # 计算 Eta Squared
        # 根据效应量和p值确定显著性标记
        if eta_squared < 0.01:
            significance = " (ns)"
        else:
            if p_val <= 0.001:
                significance = " (***)"
            elif p_val <= 0.01:
                significance = " (**)"
            elif p_val <= 0.05:
                significance = " (*)"
            else:
                significance = " (ns)"
        # 绘制 boxplot
        sns.boxplot(
            data=data_subset, x="Age_group", y="percentage",
            order=age_order, palette="Set2", width=0.6  # 颜色&箱体宽度&去掉异常值showfliers=False
        )
        # 标注标题（包含显著性标记）
        plt.title(f"{cell}{significance}", fontsize=14, fontweight="bold")
        plt.xticks(fontsize=12)
        plt.xlabel("Age Group", fontsize=12)
        plt.ylabel("Cell Type Percentage", fontsize=12)
        # 获取当前坐标轴的范围
        x_min, x_max = plt.xlim()
        y_min, y_max = plt.ylim()
        # 在图形中间位置添加 p 值和 η²
        plt.text((x_min + x_max) / 2, y_max * 0.95, 
                f"p = {p_val:.3e}, η² = {eta_squared:.3f}", 
                ha='center', fontsize=12, color='black', weight='bold')
    
        sns.despine()
        # 显示图像
        plt.show()



In [ ]:
adata.obs['age_group'].unique()


In [ ]:
meta_data = adata.obs
meta_data = meta_data[meta_data['age_group'].isin(['A', 'B', 'C', 'D', 'E', 'F', 'G'])]


In [ ]:
age_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
plot_box(meta_data, 'age_group', age_order, 'Celltype_general')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4.5, 4))
sc.pl.umap(adata, color =['Celltype'],show=True, ax=ax,ncols=1,size=3)  # 
plt.tight_layout()
plt.show()


In [ ]:
adata.X.min()


In [ ]:
adata[0:10,0:100].to_df()


In [ ]:

sc.pl.umap(adata, color=['tissue','leiden','Celltype_general', 'Celltype_sub', 'Celltype'], show=True, ncols =2, wspace=0.3 , hspace=0.2)  # wspace=0.3,legend_loc ='on data'



In [ ]:
adata.obs


In [ ]:
adata.obs['age_group'].value_counts()


In [ ]:
# adata.obs['Age_group'].value_counts()


In [ ]:
# sorted(list(adata.obs['age'].unique()))


In [ ]:
############ 然后再 assign age group    ########
def assign_age_group(age):
    # age = str(age).lower()
    if 'Young adult' in age:
        return 'B'
    elif 'nfant' in age:
        return 'A'
    elif 'E' in age:
        return 'A'
    elif age == 'M2':
        return 'A'
    elif age == 'Aged':
        return 'H'
    elif 'month' in age:
        return 'A'
    try:
        age = float(age)  # 转换为浮动类型
        if 0<=age<20:
            return 'A'
        elif 20 <= age < 30:
            return 'B'
        elif 30 <= age < 40:
            return 'C'
        elif 40 <= age < 50:
            return 'D'
        elif 50 <= age < 60:
            return 'E'
        elif 60 <= age < 70:
            return 'F'
        elif 70 <= age < 90:
            return 'G'
        elif age >= 90:
            return 'H'
    except ValueError:
        return 'Unknown'  # 对age 信息重新进行分群


In [ ]:
adata.obs['Age_group']  = adata.obs['age'].apply(assign_age_group)


In [ ]:
adata.obs['Age_group'].value_counts()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
sc.pl.umap(adata, color =['leiden'],show=True, ax=ax,ncols=1,use_raw=True, legend_loc='on data',size=3)
plt.tight_layout()
plt.show()


In [ ]:
adata.obs.columns


In [ ]:
adata.write('external_inputs/Endo_human_EndoDEG_count_notAge_633769.h5ad')


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

color_list = ['tissue', 'leiden', 'Celltype_general', 'Celltype_sub', 'Celltype']
n_colors = len(color_list)
n_cols = 2
n_rows = (n_colors + n_cols - 1) // n_cols

# 创建子图，设置每个子图为方形
fig, axes = plt.subplots(n_rows, n_cols, 
                         figsize=(n_cols * 6, n_rows * 3),  # 每个子图5x5英寸
                         squeeze=False)  # 确保axes总是2D数组

# 展平axes数组以便迭代
axes_flat = axes.flatten()

# 在每个子图上绘制UMAP
for i, color in enumerate(color_list):
    if i < len(axes_flat):
        sc.pl.umap(adata, color=color, ax=axes_flat[i], show=False)
        # 设置每个子图的标题（可选）
        axes_flat[i].set_title(color, fontweight='bold')

# 隐藏多余的子图
for i in range(len(color_list), len(axes_flat)):
    axes_flat[i].set_visible(False)

# 调整布局
plt.tight_layout()
plt.show()


In [ ]:
sc.pl.umap(adata, color =['Age_group'],show=True,ncols=1,use_raw=True,size=3)


In [ ]:
# 排序（从大到小）
import math
lengths = Fibro_meta['tissue'].value_counts()
lengths.index = [lengths.index[i]+f" ({lengths.values[i]})" for i in range(len(lengths))]
lengths = lengths.map(math.log10).sort_values(ascending=True)

# 画水平条形图
fig, ax = plt.subplots(figsize=(4, 6))
lengths.plot.barh(ax=ax, color="#7895C1")

# 设置标签和标题
ax.set_xlabel("Count / log10")
ax.set_title("Fibro cellNum distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("")

# 加粗边框
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
plt.show()
plt.savefig('./TabulaStromal_figures/Fibro_tissue_cellNum_summary.png',  dpi=300, bbox_inches='tight', format='png')



In [ ]:
os.getcwd()


In [ ]:
cell_marker_dict = {
    "SMC": [
        "MYH11",
        "TAGLN",
        "RERGL",
        "ACTA2",
        "TPM2",
        "MYL9",
        "SOD3",
        "PLN"
    ],
    "PC": [
        "COL1A1",
        "COL3A1",
        "COL1A2",
        "RGS5",
        "NDUFA4L2",
        "COL6A1",
        "THY1",
        "COL6A3",
        "BGN"
    ],
    "LEC": [
        "CCL21",
        "TFF3",
        "MMRN1",
        "EFEMP1",
        "LYVE1",
        "PROX1"
    ],
    "VenEC": [
        "ACKR1",
        "CLU",
        "CCL14",
        "SELE",
    ],
    "CapEC": [
        "RGCC",
        "ESM1",
        "INSR",
        "VWA1",
        "FLT1",
        "PLVAP",
        "HSPG2",
        "PCDH17",
        "COL15A1"
    ],
    "ArtEC": [
        "SEMA3G",
        "CXCL12",
        "FBLN5",
        "IGFBP3",
        "GJA5",
        "SRGN",
        "ARL15",
    ]
}


In [ ]:
######################  产 leiden_subtype_Short_scores.png & leiden_subtype_Short_scores.pdf, 看每个cluster的
###############  先画 6个 Lineage的, 这个是在已经做好Celltype annotation后的验证的画图
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype', 'Myfibro1_4raw', 'iFibro1_4raw',
                        'eFibro1_4raw','qFibro1_4raw', 'Myfibro_4raw', 'iFibro_4raw', 
                        'eFibro_4raw', 'qFibro_4raw']].copy()
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)  
# 定义leiden到Celltype的映射字典
# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['Myfibro_4raw', 'iFibro_4raw', 'eFibro_4raw', 'qFibro_4raw']
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows = 4
n_cols = 10
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors = plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5*n_cols, 2*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of subFibro markers',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)
# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=0
        )
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    # 隐藏x轴刻度
    ax.set_xticks([])
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Signature Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'iFibro':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'eFibro':  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'Myfibro':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    elif celltype_prefix == 'qFibro':  # qFibro
        ax.set_facecolor('#E6FFE6')  # 浅绿色背景

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
feature_short_names = []
for col in available_columns:
    if 'Myfibro1_' in col:
        feature_short_names.append('Myfibro')
        # feature_short_names.append('Myfibro_l')
    elif 'iFibro1_' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro1_' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro1_' in col:
        feature_short_names.append('qFibro')
    elif 'Myfibro_' in col:
        feature_short_names.append('Myfibro')
    elif 'iFibro_' in col:
        feature_short_names.append('iFibro')
    elif 'eFibro_' in col:
        feature_short_names.append('eFibro')
    elif 'qFibro_' in col:
        feature_short_names.append('qFibro')
    else:
        feature_short_names.append(col.replace('_4raw', ''))

# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]
# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('iFibro', '#FFF5E6'),
    2: ('eFibro', '#E6F7FF'),
    3: ('Myfibro', '#F0E6FF'),
    4: ('qFibro', '#E6FFE6')
}
# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []
for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.25, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.75, 0.01),  # 底部右侧
           ncol=4,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.1,left=0.05, right=0.9, hspace=0.4, wspace=0.3)
# 12. 保存图形
fig_path = './TabulaStromal_figures/Endothelial/'
plt.savefig(
    fig_path+"leiden_subtype_Short_scores.pdf",
    dpi=300,
    bbox_inches='tight',
    format='pdf')
fig.savefig(fig_path+'leiden_subtype_Short_scores.png', dpi=300, bbox_inches='tight')
plt.show()




In [ ]:
adata.shape



In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors
import matplotlib.patches as mpatches

# 您的细胞标记字典
cell_marker_dict = {
    "SMC": ["MYH11", "TAGLN", "RERGL", "ACTA2", "TPM2", "MYL9", "SOD3", "PLN"],
    "PC": ["COL1A1", "COL3A1", "COL1A2", "RGS5", "NDUFA4L2", "COL6A1", "THY1", "COL6A3", "BGN"],
    "LEC": ["CCL21", "TFF3", "MMRN1", "EFEMP1", "LYVE1", "PROX1"],
    "VenEC": ["ACKR1", "CLU", "CCL14", "SELE"],
    "CapEC": ["RGCC", "ESM1", "INSR", "VWA1", "FLT1", "PLVAP", "HSPG2", "PCDH17", "COL15A1"],
    "ArtEC": ["SEMA3G", "CXCL12", "FBLN5", "IGFBP3", "GJA5", "SRGN", "ARL15"]
}

def create_correct_dotplot(adata, cell_marker_dict, celltype_column='Celltype', figsize=(16, 10)):
    """
    创建正确的分组dotplot，x轴按字符串顺序排列
    """
    # 首先确保Celltype列是category类型，并按字母顺序排序
    celltypes = adata.obs[celltype_column].unique()
    celltypes_sorted = sorted(celltypes)  # 按字符串顺序排序
    
    # 将Celltype列转换为category类型并设置顺序
    adata.obs[celltype_column] = pd.Categorical(
        adata.obs[celltype_column], 
        categories=celltypes_sorted,
        ordered=True
    )
    
    print(f"X轴顺序 (按字符串排序): {celltypes_sorted}")
    
    # 准备基因列表（按组排序）
    all_genes = []
    gene_groups = []
    group_colors = {}
    
    # 为每个组分配颜色
    groups = list(cell_marker_dict.keys())
    color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    for i, (group, genes) in enumerate(cell_marker_dict.items()):
        available_genes = [gene for gene in genes if gene in adata.var_names]
        if available_genes:
            all_genes.extend(available_genes)
            gene_groups.extend([group] * len(available_genes))
            group_colors[group] = color_palette[i % len(color_palette)]
    
    if not all_genes:
        print("没有找到可用的基因")
        return
    
    # 创建基因信息DataFrame
    gene_info = pd.DataFrame({
        'gene': all_genes,
        'group': gene_groups
    })
    
    # 使用scanpy创建dotplot
    dp = sc.pl.DotPlot(adata, 
                       var_names=all_genes,
                       groupby=celltype_column,
                       standard_scale='var',  # 按基因标准化
                       dot_max=0.7,          # 最大点大小
                       dot_min=0.1,          # 最小点大小
                       cmap='Reds',          # 颜色映射
                       figsize=figsize)
    
    # 自定义样式
    dp.style(edgecolor='black', linewidth=0.5)
    
    # 获取图形和坐标轴
    fig = dp.get_figure()
    ax = fig.axes[0]
    
    # 添加分组分隔线
    current_group = None
    for i, gene in enumerate(all_genes):
        group = gene_info[gene_info['gene'] == gene]['group'].iloc[0]
        if group != current_group and current_group is not None:
            ax.axhline(i - 0.5, color='gray', linestyle='-', linewidth=2, alpha=0.7)
        current_group = group
    
    # 添加组标签到y轴
    ytick_labels = []
    current_group = None
    group_positions = {}
    
    for i, gene in enumerate(all_genes):
        group = gene_info[gene_info['gene'] == gene]['group'].iloc[0]
        if group != current_group:
            # 记录组的起始位置
            if group not in group_positions:
                group_positions[group] = i
            ytick_labels.append(f"{gene}\n({group})")
            current_group = group
        else:
            ytick_labels.append(gene)
    
    ax.set_yticklabels(ytick_labels, fontsize=10)
    
    # 创建图例
    legend_patches = [mpatches.Patch(color=color, label=group) 
                     for group, color in group_colors.items()]
    
    # 在主图旁边添加图例
    ax.legend(handles=legend_patches, 
              loc='center left', 
              bbox_to_anchor=(1.05, 0.5),
              frameon=True,
              fancybox=True,
              shadow=True,
              title='Cell Types')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# 更简单的方法：直接使用scanpy，通过基因排序实现分组
def create_simple_correct_dotplot(adata, cell_marker_dict, celltype_column='Celltype', figsize=(14, 8)):
    """
    简单版本：通过基因排序实现分组效果，x轴按字符串顺序排列
    """
    # 首先确保Celltype列是category类型，并按字母顺序排序
    celltypes = adata.obs[celltype_column].unique()
    celltypes_sorted = sorted(celltypes)  # 按字符串顺序排序
    
    # 将Celltype列转换为category类型并设置顺序
    adata.obs[celltype_column] = pd.Categorical(
        adata.obs[celltype_column], 
        categories=celltypes_sorted,
        ordered=True
    )
    
    print(f"X轴顺序 (按字符串排序): {celltypes_sorted}")
    
    # 按组排序基因
    grouped_genes = []
    for group, genes in cell_marker_dict.items():
        available_genes = [gene for gene in genes if gene in adata.var_names]
        grouped_genes.extend(available_genes)
    
    if not grouped_genes:
        print("没有找到可用的基因")
        return
    
    # 创建dotplot
    fig = sc.pl.dotplot(adata,
                        var_names=cell_marker_dict,
                       # var_names=grouped_genes,
                       groupby=celltype_column,
                       standard_scale='var',
                       dot_max=0.7,
                       dot_min=0.1,
                       cmap='Reds',
                       figsize=figsize,
                       show=False,
                       return_fig=True)
    
    # 获取坐标轴
    ax = fig.axes[0] if hasattr(fig, 'axes') else plt.gca()
    
    # 添加分组分隔线
    current_index = 0
    for group, genes in cell_marker_dict.items():
        available_genes = [gene for gene in genes if gene in adata.var_names]
        if available_genes:
            if current_index > 0:  # 不在第一个组添加线
                ax.axhline(current_index - 0.5, color='blue', linestyle='--', linewidth=2, alpha=0.7)
            current_index += len(available_genes)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# 方法3：使用seaborn创建完全自定义的dotplot
def create_custom_seaborn_dotplot(adata, cell_marker_dict, celltype_column='Celltype', figsize=(18, 12)):
    """
    使用seaborn创建完全自定义的dotplot，x轴按字符串顺序排列
    """
    # 获取排序后的细胞类型
    celltypes = adata.obs[celltype_column].unique()
    celltypes_sorted = sorted(celltypes)
    
    print(f"X轴顺序 (按字符串排序): {celltypes_sorted}")
    
    # 准备数据
    all_data = []
    
    for group, genes in cell_marker_dict.items():
        for gene in genes:
            if gene in adata.var_names:
                # 按排序后的细胞类型顺序处理
                for celltype in celltypes_sorted:
                    # 获取该细胞类型的细胞索引
                    cell_mask = adata.obs[celltype_column] == celltype
                    gene_expression = adata[cell_mask, gene].X
                    
                    # 处理稀疏矩阵
                    if hasattr(gene_expression, 'toarray'):
                        gene_expression = gene_expression.toarray().flatten()
                    else:
                        gene_expression = gene_expression.flatten()
                    
                    # 计算平均表达和表达比例
                    mean_expr = np.mean(gene_expression)
                    frac_expr = np.mean(gene_expression > 0)
                    
                    all_data.append({
                        'Gene': gene,
                        'CellType': celltype,
                        'Group': group,
                        'MeanExpression': mean_expr,
                        'FractionExpressed': frac_expr
                    })
    
    if not all_data:
        print("没有找到可用的数据")
        return
    
    plot_df = pd.DataFrame(all_data)
    
    # 确保CellType按排序后的顺序
    plot_df['CellType'] = pd.Categorical(
        plot_df['CellType'], 
        categories=celltypes_sorted,
        ordered=True
    )
    
    # 创建图形
    fig, ax = plt.subplots(figsize=figsize)
    
    # 创建点图
    scatter = ax.scatter(
        x=plot_df['CellType'],
        y=plot_df['Gene'],
        s=plot_df['FractionExpressed'] * 200,  # 点大小表示表达比例
        c=plot_df['MeanExpression'],           # 点颜色表示平均表达量
        cmap='Reds',
        edgecolor='black',
        linewidth=0.5
    )
    
    # 设置坐标轴
    ax.set_xlabel('Cell Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('Gene', fontsize=12, fontweight='bold')
    ax.set_title('Cell Type Marker Expression Dotplot', fontsize=14, fontweight='bold')
    
    # 旋转x轴标签
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    # 添加颜色条
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Mean Expression', fontsize=10)
    
    # 添加分组分隔线
    current_group = None
    genes = plot_df['Gene'].unique()
    for i, gene in enumerate(genes):
        group = plot_df[plot_df['Gene'] == gene]['Group'].iloc[0]
        if group != current_group and current_group is not None:
            ax.axhline(i - 0.5, color='gray', linestyle='-', linewidth=2, alpha=0.7)
        current_group = group
    
    plt.tight_layout()
    plt.show()
    
    return fig


original_celltypes = adata.obs['Celltype'].unique()


In [ ]:

# 方法2：简单版本
try:
    fig2 = create_simple_correct_dotplot(adata, cell_marker_dict, 'Celltype', figsize=(14, 8))
    fig2.savefig('simple_celltype_dotplot_sorted.png', dpi=300, bbox_inches='tight')
    print("方法2 - dotplot已保存")
except Exception as e:
    print(f"方法2失败: {e}")

# 方法3：完全自定义版本
try:
    fig3 = create_custom_seaborn_dotplot(adata, cell_marker_dict, 'Celltype', figsize=(18, 12))
    fig3.savefig('custom_celltype_dotplot_sorted.png', dpi=300, bbox_inches='tight')
    print("方法3 - dotplot已保存")
except Exception as e:
    print(f"方法3失败: {e}")



In [ ]:
meta_data = adata.obs


In [ ]:
# 提取tissue和GSM列
tissue_gsm_df = adata.obs[["tissue", "GSM"]].copy()

#统计各组织的细胞数和原始GSM数
tissue_cell_count = tissue_gsm_df.groupby("tissue").size().reset_index(name="cell_count")
tissue_gsm_count = tissue_gsm_df.groupby("tissue")["GSM"].nunique().reset_index(name="gsm_count")

# 合并数据
tissue_stats = pd.merge(tissue_cell_count, tissue_gsm_count, on="tissue", how="outer")
tissue_stats = tissue_stats.dropna(subset=["cell_count", "gsm_count"]).reset_index(drop=True)
# 转换数据类型
tissue_stats["cell_count"] = tissue_stats["cell_count"].astype(int)
tissue_stats["gsm_count"] = tissue_stats["gsm_count"].astype(int)

# 按细胞数降序排列
custom_tissue_order = tissue_stats.sort_values("cell_count", ascending=False)["tissue"].tolist()
tissue_stats = tissue_stats[tissue_stats["tissue"].isin(custom_tissue_order)].copy()
tissue_stats["tissue"] = pd.Categorical(tissue_stats["tissue"], categories=custom_tissue_order, ordered=True)
tissue_stats = tissue_stats.sort_values("tissue").reset_index(drop=True)

#GSM负向高度放大500倍
zoom_factor = 500  # 固定放大倍数
tissue_stats["gsm_count_neg_zoom"] = -tissue_stats["gsm_count"] * zoom_factor  # 负向显示

# 打印统计结果
print("各组织的细胞数（降序）、实际GSM数与放大后柱子高度：")
print(tissue_stats[["tissue", "cell_count", "gsm_count", "gsm_count_neg_zoom"]])
print(f"\nGSM放大倍数：{zoom_factor}倍（GSM柱子高度=实际GSM数×{zoom_factor}）")

plt.figure(figsize=(14, 8))  # 图表尺寸
# 颜色
cell_color = "#2E86AB"  # 细胞数（蓝色系）
gsm_color = "#A23B72"   # GSM数（紫红色系）

# 细胞数（向上）
bars_cell = plt.bar(
    x=tissue_stats["tissue"],  # x轴为组织类别（细胞数降序）
    height=tissue_stats["cell_count"],  # y轴为细胞数
    color=cell_color,
    alpha=0.8,
    label="Cell Number"  # 图例
)

# GSM数（向下，放大500倍）
bars_gsm = plt.bar(
    x=tissue_stats["tissue"],  # x轴与细胞数对齐
    height=tissue_stats["gsm_count_neg_zoom"],  # y轴为负向放大后的GSM高度
    color=gsm_color,
    alpha=0.8,
    label="Sample Number"  # 图例
)


# 修改y轴刻度（负向显示实际GSM数，正向显示原始细胞数）
ax = plt.gca()  # 获取当前坐标轴对象
y_ticks = ax.get_yticks()  # 自动生成的y轴刻度值

custom_y_labels = []
for tick in y_ticks:
    if tick < 0:  # 选中负向刻度
        actual_gsm = int(tick / (-zoom_factor))  # 还原为实际GSM数（刻度值 ÷ (-放大倍数)）
        custom_y_labels.append(str(actual_gsm))
    elif tick == 0:  # 0刻度：保持“0”
        custom_y_labels.append("0")
    else:  # 选中正向刻度：保留原始数值并添加千分位
        custom_y_labels.append(f"{int(tick):,}")

# 应用自定义刻度标签
ax.set_yticklabels(custom_y_labels, fontsize=9)

plt.xticks(rotation=-45, ha='left', fontsize=9)  # 保持x轴标签旋转

# 细胞数标注（显示各组织的实际细胞数，带千分位）
for bar, count in zip(bars_cell, tissue_stats["cell_count"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,  # 标注水平居中（柱子中点）
        bar.get_height() + tissue_stats["cell_count"].max() * 0.01,  # 标注在柱子顶部上方（避免重叠）
        s=f"{count:,}",  # 实际细胞数
        ha="center", va="bottom",  # 对齐方式
        fontsize=9,
        fontweight="bold"
    )

# GSM数标注
for bar, count in zip(bars_gsm, tissue_stats["gsm_count"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,  # 标注水平居中
        bar.get_height() - (tissue_stats["gsm_count"].max() * zoom_factor) * 0.02,  # 标注在柱子底部下方
        s=f"{count}",  # 实际GSM数
        ha="center", va="top",  # 对齐方式
        fontsize=9,
        fontweight="bold",
        color=gsm_color  # 标注颜色
    )

plt.xlabel("Tissue Type ", fontsize=12, labelpad=10)  
plt.ylabel("Count", fontsize=11, labelpad=10)  
plt.title("Distribution of Cell Number and Sample Number of Different Tissue Types ", fontsize=14, pad=20)  

plt.legend(loc="upper center", fontsize=10, bbox_to_anchor=(0.5, 0.7), ncol=2)  # 图例位置
plt.axhline(y=0, color="black", linestyle="-", linewidth=2)  # 加粗0刻度线

# 调整布局
plt.tight_layout()
plt.show()


In [ ]:
######## 
# 排序（从大到小）
import math
lengths = Fibro_meta['tissue'].value_counts()
lengths.index = [lengths.index[i]+f" ({lengths.values[i]})" for i in range(len(lengths))]
lengths = lengths.map(math.log10).sort_values(ascending=True)

# 画水平条形图
fig, ax = plt.subplots(figsize=(4, 6))
lengths.plot.barh(ax=ax, color="#7895C1")

# 设置标签和标题
ax.set_xlabel("Count / log10")
ax.set_title("Fibro cellNum distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("")

# 加粗边框
for spine in ax.spines.values():
    spine.set_linewidth(2)

plt.tight_layout()
plt.show()
plt.savefig('./TabulaStromal_figures/Fibro_tissue_cellNum_summary.png',  dpi=300, bbox_inches='tight', format='png')



In [ ]:
Fibro_meta


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math

# 准备数据
Fibro_meta_sample = meta_data[['sample', 'tissue', 'age']].drop_duplicates()
sample_counts = Fibro_meta_sample['tissue'].value_counts()
cell_counts = meta_data['tissue'].value_counts()

# 确保两个series的index顺序一致
tissues = sample_counts.index
sample_counts = sample_counts.loc[tissues]
cell_counts = cell_counts.loc[tissues]

# 对数据进行排序（按sample数从大到小）
cell_counts = cell_counts.sort_values(ascending=False)
sample_counts = sample_counts.loc[cell_counts.index]

# 应用log10转换
sample_counts_log = sample_counts.map(math.log10)
cell_counts_log = cell_counts.map(math.log10)

# 创建图形
fig, ax = plt.subplots(figsize=(12, 12))

# 设置y轴位置
y_pos = np.arange(len(tissues))

# 绘制两个横向柱状图（对数刻度）
bar_width = 0.35

bars1 = ax.barh(y_pos - bar_width/2, sample_counts_log, bar_width, 
                label='Sample(log10)',color="#E3625D", alpha=0.8)
bars2 = ax.barh(y_pos + bar_width/2, cell_counts_log, bar_width, 
                label='Cell(log10)',color="#7895C1", alpha=0.8)

# 设置y轴标签
ax.set_yticks(y_pos)
y_labels = [f"{tissue}(S:{sample_counts[tissue]:,} | C:{cell_counts[tissue]:,})" 
            for tissue in sample_counts.index]
ax.set_yticklabels(y_labels)

# 设置x轴标签和标题
ax.set_xlabel('Count (log10)', fontsize=12, fontweight='bold')
ax.set_title('Endothelial tissue distribution: Sample & Cell number(Log10)', 
             fontsize=14, fontweight='bold')

# 添加图例
ax.legend(loc='lower right', frameon=True, fancybox=True, shadow=True)

# 添加数值标签（显示原始数值，不是对数值）
def add_value_labels_log(ax, bars, original_values):
    """在对数刻度柱子上添加原始数值标签"""
    for i, bar in enumerate(bars):
        width = bar.get_width()
        if width > 0:
            original_value = original_values[i]
            ax.text(width + 0.05, bar.get_y() + bar.get_height()/2,
                   f'{original_value:,}',  # 显示原始数值
                   ha='left', va='center', fontsize=9, fontweight='bold')

# 添加数值标签
add_value_labels_log(ax, bars1, sample_counts.values)
add_value_labels_log(ax, bars2, cell_counts.values)

# 加粗边框
for spine in ax.spines.values():
    spine.set_linewidth(2)

# 添加网格线
ax.grid(True, axis='x', alpha=0.3, linestyle='--')

# 调整布局
plt.tight_layout()

# # 保存图片
# plt.savefig('./TabulaStromal_figures/Fibro_tissue_sample_cell_summary_log.png', 
#             dpi=300, bbox_inches='tight', format='png')

plt.show()



In [ ]:
score_dict = {
    "antigen_pre": [
        "CD74",
        "HLA-DQA1",
        "CD83",
        "HLA-DRA",
        "HLA-DPA1",
        "HLA-DQA",
        "HLA-DRB1",
        "H2-Q4"
    ],
    "Tip": [
        "ANGPT2",
        "APLN",
        "FSCN1",
        "PGF",
        "PLXND1",
        "ADM",
        "PDGFB",
        "CXCR4",
        "ESM1",
        "PXDN",
        "LXN",
        "ANGPTL2",
        "INSR"
    ],
    "stalk": [
        "ACKR1",
        "SELE",
        "CLU",
        "ADIRF",
        "CCL2",
        "IL33",
        "POSTN",
        "HLA-DRA"
    ]
}


In [ ]:
leiden_score_df.head(10)

In [ ]:

score_cols = [f"score_{gs_name}" for gs_name in score_dict.keys()]
leiden_score_df = adata.obs[["Celltype"] + score_cols].copy()
leiden_avg_score = leiden_score_df.groupby("Celltype").agg({col: "mean" for col in score_cols}).reset_index()

# 定义3个基因集得分列
score1, score2, score3 = score_cols
score1_name = score1.replace("score_", "") 
score2_name = score2.replace("score_", "")
score3_name = score3.replace("score_", "")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 子图1
ax1.scatter(
    leiden_avg_score[score1],  # x轴：第一个基因集得分
    leiden_avg_score[score2],  # y轴：第二个基因集得分
    #c=leiden_avg_score["Celltype"],  # 用簇编号着色
    cmap="viridis",  # 配色方案
    s=80,  # 点大小
    alpha=0.7  # 透明度
)
# 标注簇编号
for _, row in leiden_avg_score.iterrows():
    ax1.text(row[score1], row[score2], s=row["Celltype"], fontsize=7, ha="center")
# 轴标签和标题
ax1.set_xlabel(score1_name, fontsize=10)
ax1.set_ylabel(score2_name, fontsize=10)
ax1.set_title(f"{score1_name} vs {score2_name}", fontsize=11)
ax1.grid(alpha=0.3)  # 显示网格辅助读值

# 子图2
ax2.scatter(
    leiden_avg_score[score1],
    leiden_avg_score[score3],
    #c=leiden_avg_score["Celltype"],
    cmap="viridis",
    s=80,
    alpha=0.7
)
# 标注簇编号
for _, row in leiden_avg_score.iterrows():
    ax2.text(row[score1], row[score3], s=row["Celltype"], fontsize=7, ha="center")
# 轴标签和标题
ax2.set_xlabel(score1_name, fontsize=10)
ax2.set_ylabel(score3_name, fontsize=10)
ax2.set_title(f"{score1_name} vs {score3_name}", fontsize=11)
ax2.grid(alpha=0.3)

# 子图3
ax3.scatter(
    leiden_avg_score[score2],
    leiden_avg_score[score3],
    #c=leiden_avg_score["Celltype"],
    cmap="viridis",
    s=80,
    alpha=0.7
)
# 标注簇编号
for _, row in leiden_avg_score.iterrows():
    ax3.text(row[score2], row[score3], s=row["Celltype"], fontsize=7, ha="center")
# 轴标签和标题
ax3.set_xlabel(score2_name, fontsize=10)
ax3.set_ylabel(score3_name, fontsize=10)
ax3.set_title(f"{score2_name} vs {score3_name}", fontsize=11)
ax3.grid(alpha=0.3)

# 调整子图间距
plt.tight_layout()
# 显示图片
plt.show()

# --------------------------
# 1. 数据准备：按Leiden簇计算均值（统一为第一段逻辑）+ 关联细胞类型
# --------------------------
# 生成基因集得分列名（与原逻辑一致）
score_cols = [f"score_{gs_name}" for gs_name in score_dict.keys()]

# 步骤1：先按第一段逻辑，计算每个Leiden簇的基因集得分均值（核心统一点）
# 提取Leiden簇和基因集得分列
leiden_score_df = adata.obs[["Celltype"] + score_cols].copy()
# 按Leiden簇分组，计算每个基因集得分的均值（与第一段完全一致的均值逻辑）
leiden_avg_score = leiden_score_df.groupby("Celltype").agg({col: "mean" for col in score_cols}).reset_index()

# 关联细胞类型
# 提取Leiden簇与细胞类型的对应关系
leiden_celltype =adata.obs[[ "Celltype"]].drop_duplicates(subset="Celltype").reset_index(drop=True)  # adata.obs[["leiden", "Celltype"]].drop_duplicates(subset="leiden").reset_index(drop=True)

# 合并“簇均值得分”和“细胞类型”（确保每个Leiden簇对应唯一得分和细胞类型）
leiden_score_with_celltype = leiden_avg_score.merge(
    leiden_celltype, 
    on="Celltype",  # 按Leiden簇编号关联
    how="inner"   # 只保留两边都存在的簇（避免空值）
)

# 定义3个基因集得分列
score1, score2, score3 = score_cols  # 分别对应Antigen_pre、Tip、Stalk（按score_dict顺序）
score1_name = score1.replace("score_", "")  
score2_name = score2.replace("score_", "")
score3_name = score3.replace("score_", "")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.scatter(
    leiden_score_with_celltype[score1],  
    leiden_score_with_celltype[score2],
    #c=leiden_score_with_celltype["Celltype"],  # 颜色仍按Leiden簇编号
    cmap="viridis",
    s=80,
    alpha=0.7
)
# 标注：显示细胞类型
for _, row in leiden_score_with_celltype.iterrows():
    ax1.text(
        row[score1], row[score2],
        s=row["Celltype"],
        fontsize=7,
        ha="center"
    )
ax1.set_xlabel(score1_name, fontsize=10)
ax1.set_ylabel(score2_name, fontsize=10)
ax1.set_title(f"{score1_name} vs {score2_name}", fontsize=11)
ax1.grid(alpha=0.3)

# 子图2
ax2.scatter(
    leiden_score_with_celltype[score1],
    leiden_score_with_celltype[score3],
    #c=leiden_score_with_celltype["Celltype"],
    cmap="viridis",
    s=80,
    alpha=0.7
)
for _, row in leiden_score_with_celltype.iterrows():
    ax2.text(
        row[score1], row[score3],
        s=row["Celltype"],
        fontsize=7,
        ha="center"
    )
ax2.set_xlabel(score1_name, fontsize=10)
ax2.set_ylabel(score3_name, fontsize=10)
ax2.set_title(f"{score1_name} vs {score3_name}", fontsize=11)
ax2.grid(alpha=0.3)

# 子图3
ax3.scatter(
    leiden_score_with_celltype[score2],
    leiden_score_with_celltype[score3],
    # c=leiden_score_with_celltype["Celltype"],    # .astype(int)
    cmap="viridis",
    s=80,
    alpha=0.7
)
for _, row in leiden_score_with_celltype.iterrows():
    ax3.text(
        row[score2], row[score3],
        s=row["Celltype"],
        fontsize=7,
        ha="center"
    )
ax3.set_xlabel(score2_name, fontsize=10)
ax3.set_ylabel(score3_name, fontsize=10)
ax3.set_title(f"{score2_name} vs {score3_name}", fontsize=11)
ax3.grid(alpha=0.3)

fig.suptitle("Leiden Clusters (Labeled by Cell Type) vs Gene Set Scores", fontsize=13, y=1.02)
plt.show()


In [ ]:
len(adata.obs['leiden'].unique())


In [ ]:
score1, score2, score3


In [ ]:
leiden_score_with_celltype.shape

In [ ]:
leiden_score_with_celltype.head(10)


In [ ]:
leiden_avg_score.head(10)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# 创建图形
fig = plt.figure(figsize=(18, 14))
ax = fig.add_subplot(111, projection='3d')

# 提取数据
x = leiden_avg_score['score_antigen_pre'].values
y = leiden_avg_score['score_Tip'].values
z = leiden_avg_score['score_stalk'].values
celltypes = leiden_avg_score['Celltype'].values

# 生成颜色
n_colors = len(leiden_avg_score)
color_list = plt.cm.tab20(np.linspace(0, 1, min(20, n_colors))).tolist()
if n_colors > 20:
    additional_colors = plt.cm.Set3(np.linspace(0, 1, n_colors - 20)).tolist()
    color_list.extend(additional_colors)

# 绘制散点图 - 使用不同大小区分密集区域
# 计算每个点与其他点的平均距离，密集区域的点画小一点
from scipy.spatial import distance

points = np.column_stack([x, y, z])
avg_distances = []
for i in range(len(points)):
    dists = distance.cdist([points[i]], points)[0]
    avg_distances.append(np.mean(dists))

# 根据密集程度调整点大小
point_sizes = np.interp(avg_distances, 
                       [min(avg_distances), max(avg_distances)], 
                       [80, 150])

scatter = ax.scatter(x, y, z, 
                    c=color_list,
                    s=point_sizes,
                    alpha=0.8,
                    edgecolors='black',
                    linewidth=0.8)

# 扩展坐标轴范围（增加30%的边距）
x_range = x.max() - x.min()
y_range = y.max() - y.min()
z_range = z.max() - z.min()

x_margin = x_range * 0.3
y_margin = y_range * 0.3
z_margin = z_range * 0.3

ax.set_xlim(x.min() - x_margin, x.max() + x_margin)
ax.set_ylim(y.min() - y_margin, y.max() + y_margin)
ax.set_zlim(z.min() - z_margin, z.max() + z_margin)

# 更智能的文本偏移函数
def get_smart_text_offset(i, x_val, y_val, z_val, all_points, base_offset=0.4):
    """根据点的位置智能选择偏移方向"""
    # 计算点的大致象限
    x_center = np.mean(all_points[:, 0])
    y_center = np.mean(all_points[:, 1])
    z_center = np.mean(all_points[:, 2])
    
    # 根据点相对于中心的位置决定偏移方向
    x_dir = 1 if x_val < x_center else -1
    y_dir = 1 if y_val < y_center else -1
    z_dir = 1 if z_val < z_center else -1
    
    # 添加一些随机性避免完全对称
    random_factor = 0.1
    offset_x = x_dir * base_offset * (1 + random_factor * (i % 3))
    offset_y = y_dir * base_offset * (1 + random_factor * ((i + 1) % 3))
    offset_z = z_dir * base_offset * (1 + random_factor * ((i + 2) % 3))
    
    return offset_x, offset_y, offset_z

# 添加文本标注 - 使用智能偏移
for i, celltype in enumerate(celltypes):
    offset_x, offset_y, offset_z = get_smart_text_offset(i, x[i], y[i], z[i], points, base_offset=0.6)
    
    # 根据文本长度调整字体大小
    text_size = 6 if len(celltype) > 15 else 7
    
    ax.text(x[i] + offset_x, y[i] + offset_y, z[i] + offset_z, 
            celltype, 
            fontsize=text_size, 
            ha='left' if offset_x > 0 else 'right',
            va='bottom' if offset_y > 0 else 'top',
            bbox=dict(boxstyle='round,pad=0.15', 
                     facecolor='lightyellow', 
                     alpha=0.85, 
                     edgecolor='gray',
                     linewidth=0.3))

# 设置标签和标题
ax.set_xlabel('Score Antigen Pre', fontsize=13, fontweight='bold', labelpad=18)
ax.set_ylabel('Score Tip', fontsize=13, fontweight='bold', labelpad=18)
ax.set_zlabel('Score Stalk', fontsize=13, fontweight='bold', labelpad=18)
ax.set_title('3D Visualization of Cell Types - Improved Layout', 
             fontsize=16, fontweight='bold', pad=25)

# 添加网格
ax.grid(True, alpha=0.3)

# 设置多个视角并保存
angles = [(25, 45), (15, 135), (30, 225)]  # 不同的视角

for j, (elev, azim) in enumerate(angles):
    ax.view_init(elev=elev, azim=azim)
    plt.tight_layout()
    # plt.savefig(f'3d_scatter_celltypes_view_{j+1}.pdf', 
    #             dpi=300, 
    #             bbox_inches='tight', 
    #             format='pdf')
    print(f"已保存视角 {j+1}: elev={elev}, azim={azim}")

# 显示最后一个视角
plt.show()


In [ ]:
# sorted(dir())


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# 假设您的数据框名为leiden_avg_score
# 创建示例数据（您可以直接使用您的实际数据）
# leiden_avg_score = pd.DataFrame({...})  # 您的实际数据

# 创建图形和3D坐标轴
fig = plt.figure(figsize=(11, 8.5))
ax = fig.add_subplot(111, projection='3d')

# 提取数据
x = leiden_avg_score['score_antigen_pre'].values
y = leiden_avg_score['score_Tip'].values
z = leiden_avg_score['score_stalk'].values
celltypes = leiden_avg_score['Celltype'].values

# 生成29种独特颜色（根据您的数据长度调整）
n_colors = len(leiden_avg_score)
color_list = plt.cm.tab20(np.linspace(0, 1, 20)).tolist()
if n_colors > 20:
    additional_colors = plt.cm.Set3(np.linspace(0, 1, n_colors - 20)).tolist()
    color_list.extend(additional_colors)
color_list = color_list[:n_colors]

# 绘制3D散点图
scatter = ax.scatter(x, y, z, 
                    c=color_list,
                    s=100,           # 点的大小
                    alpha=0.8,       # 透明度
                    edgecolors='black',  # 边缘颜色
                    linewidth=0.5)   # 边缘线宽

# 为每个点添加文本标注
# for i, celltype in enumerate(celltypes):
#     ax.text(x[i], y[i], z[i], 
#             f'  {celltype}', 
#             fontsize=8, 
#             ha='left', 
#             va='bottom',
#             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))

# 设置坐标轴标签
ax.set_xlabel('Antigen Presenting', fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel('Tip', fontsize=12, fontweight='bold', labelpad=10)
ax.set_zlabel('Stalk', fontsize=12, fontweight='bold', labelpad=0)

ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))

# 设置标题
# ax.set_title('Signature Score', 
#              fontsize=16, fontweight='bold', pad=20)

# 添加网格
ax.grid(False)  #  ax.grid(True, alpha=0.3)


# 设置视角（可选，调整以获得最佳视角）
ax.view_init(elev=20, azim=45)  # 仰角和方位角

# 创建图例（由于每个点颜色不同，创建自定义图例）
legend_elements = []
for i, celltype in enumerate(celltypes):
    legend_elements.append(plt.Line2D([0], [0], 
                                    marker='o', 
                                    color='w', 
                                    markerfacecolor=color_list[i],
                                    markersize=8, 
                                    label=celltype))

# 添加图例（可以调整位置避免遮挡）
ax.legend(handles=legend_elements, 
          loc='center right', 
          bbox_to_anchor=(1, 0.8),
          fontsize=8,
          ncol=2,  # 图例列数
          frameon=True,
          fancybox=True,
          shadow=True)

# 调整布局
plt.tight_layout()
# # 保存为PDF格式
plt.savefig('./TabulaStromal_figures/Endothelial/3d_scatter_celltype_score_noText.pdf', 
            dpi=300, bbox_inches='tight', format='pdf')

# 显示图形
plt.show()


In [ ]:
####### adata
help(ax.grid)


In [ ]:
adata.obs[['Celltype','Celltype_sub','Celltype_general','tissue','age','Age_group']].drop_duplicates()


In [ ]:
def isstr(x):
    try: 
        float(x)
        return False
    except ValueError as e:
        return True



In [ ]:
len(meta_data[meta_data['disease_global'].isin(['Normal','Healthy'])]['sample'].unique())


In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

# 1. 创建一个图形（Figure）和坐标轴（Axes）对象，并设置白色背景
fig, ax = plt.subplots(figsize=(7, 5), facecolor='white')  # facecolor设置图形背景为白色[citation:6]
ax.set_facecolor('white')  # 设置坐标轴区域（绘图区）背景为白色[citation:6]

# 2. 绘制UMAP散点图，颜色按Celltype区分
# 确保使用之前计算好的UMAP坐标
sc.pl.embedding(
    adata,
    basis='umap',  # 指定使用umap坐标
    color='Celltype',  # 按Celltype着色
    ax=ax,  # 指定在上面创建的坐标轴ax上绘制
    show=False,  # 不立即显示，以便后续自定义
    frameon=False,  # 关键参数：关闭绘图边框[citation:3]
    # legend_loc='on data',  # 可选：将图例放在图上，避免遮挡。也可用 'right margin'
    title=''  # 清除自动生成的标题，使图片更简洁
)

# 3. 隐藏所有默认的坐标轴边框（上、下、左、右）
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# 4. 移除默认的刻度线和刻度标签
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')  # 清空x轴标签
ax.set_ylabel('')  # 清空y轴标签

# 5. 获取当前UMAP图的坐标轴范围，用于定位箭头
x_min, x_max = ax.get_xlim()
y_min, y_max = ax.get_ylim()

# 定义箭头长度（为轴长的5%）
arrow_length_x = (x_max - x_min) * 0.05
arrow_length_y = (y_max - y_min) * 0.05

# 6. 在坐标轴末端手动添加小箭头
# 6.1 添加UMAP1轴（X轴）箭头：在右下角，指向X正方向
ax.annotate('',
            xy=(x_max, y_min),  # 箭头终点：X最大，Y最小（右下角）
            xytext=(x_max - arrow_length_x, y_min),  # 箭头起点：向回退一点
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5, shrinkA=0, shrinkB=0),
            annotation_clip=False)
# 可选：添加“UMAP1”标签
ax.text(x_max, y_min - arrow_length_y, 'UMAP1', ha='right', va='top', fontsize=10)

# 6.2 添加UMAP2轴（Y轴）箭头：在左上角，指向Y正方向
ax.annotate('',
            xy=(x_min, y_max),  # 箭头终点：X最小，Y最大（左上角）
            xytext=(x_min, y_max - arrow_length_y),  # 箭头起点：向下退一点
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5, shrinkA=0, shrinkB=0),
            annotation_clip=False)
# 可选：添加“UMAP2”标签
ax.text(x_min - arrow_length_x, y_max, 'UMAP2', ha='right', va='top', fontsize=10, rotation=90)

# 7. 将图形保存为PDF格式[citation:5][citation:8]
output_path = './code/Endo_Celltype_umap.pdf'
plt.savefig(output_path, format='pdf', bbox_inches='tight', dpi=300)
print(f"UMAP图已成功保存至：{output_path}")

# 8. 可选：在屏幕上显示图形
plt.show()


In [ ]:
adata.obs = adata.obs.drop('age_group', axis=1)
adata.obs.columns


In [ ]:

adata  = adata[~np.array(list(map(lambda x: isstr(x),adata.obs['age'])))]


In [ ]:
adata.obs['disease_global'].value_counts()


In [ ]:
adata.obs['age'] = adata.obs['age'].astype(float)


In [ ]:
# adata.obs['age'].unique()


In [ ]:
# len(adata.obs['sample'].unique())
adata.obs


In [ ]:
# sorted(adata.obs['age'].unique())


In [ ]:
# meta_data


In [ ]:
# adata = sc.read_h5ad('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')


In [ ]:
adata.write('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_GSMbatch_annotation_delNoneAge.h5ad')


In [ ]:
# adata.obs['Age_group'].value_counts()


In [ ]:
adata.obs['disease_global_check'].unique()
mask = (adata.obs['disease_global_check'] == 'nan')


In [ ]:
# adata.obs['disease_global_check'].value_counts()

In [ ]:
adata.obs.loc[mask, 'disease_global_check'] = adata.obs.loc[mask, 'disease_global']


In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()


In [ ]:
adata.obs['disease_global_check'].value_counts()


In [ ]:
adata.obs['disease_global_check'].isin(['Normal','Healthy'])


In [ ]:
adata_h.obs['Age'] = adata_h.obs['age']


In [ ]:
adata = temp_adata
del temp_adata


In [ ]:
Endo_meta_check = pd.read_csv('./data/scalex/endo/human/Endo_human_EndoDEG_count_notAge_633769_check.csv',sep=',')
Endo_meta_check


In [ ]:
Fibro_meta = pd.read_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta.tsv',sep='\t')
Fibro_meta


In [ ]:
Fibro_unique = Fibro_meta[['GSE','GSM','age','Age_group','tissue','disease_global','sex']].drop_duplicates()


In [ ]:
# Fibro_unique.to_csv('./data/scalex/fibro/human/Fibro_human_FibDEG_count_notAge_GSMbatch_annotation_meta_check.tsv',sep='\t')


    

In [ ]:
# Endo_meta_check


In [ ]:
Endo_meta_check.columns


In [ ]:
adata.shape, meta_data.shape

In [ ]:
Endo_meta_check['disease_global'].unique()



In [ ]:
Endo_meta_check['disease_global_check'].unique()



In [ ]:
adata.obs['age'].value_counts()



In [ ]:
adata.obs.shape


In [ ]:
adata.obs['disease_global_check'].value_counts()



In [ ]:
# 根据['GSE','GSM']这两列进行合并
merged_obs = adata.obs.merge(
    Endo_meta_check[['GSE', 'GSM', 'age_check', 'Age_group_check', 'tissue_check', 'disease_global_check', 'sex_check']],
    on=['GSE', 'GSM'],
    how='left'
)

# # 更新adata.obs
adata.obs = merged_obs



In [ ]:
adata.obs[['GSE','disease_global','disease_global_check']].drop_duplicates()


In [ ]:

intersection_sorted = sorted(list(set(sorted(list(adata.obs['GSM'].unique()))) & set(Endo_meta_check['GSM'].unique())))

len(intersection_sorted),len(Endo_meta_check['GSM'].unique()),len(adata.obs['GSM'].unique())

In [ ]:
# adata.obs[['GSE','GSM','Age','disease_global','disease_global_check']].drop_duplicates()


In [ ]:
adata.obs[['disease_global','disease_global_check']].drop_duplicates()


In [ ]:
adata.obs['Celltype_general'].value_counts()


In [ ]:
age_order = ['A','B', 'C', 'D', 'E', 'F', 'G']
plot_box(adata_h, 'Age_group', age_order, 'Celltype_general')


In [ ]:
adata[adata.obs['disease_global']=='Cancer']

In [ ]:
#################################### First try the random forest model 
# adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal', 'Healthy'])) & (adata.obs['Age']!=0)]
adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal','Healthy']))]  #  & (adata.obs['tissue']=='skin')
target_cell_types = list(set(adata_h.obs['Celltype']))


In [ ]:

adata_immune = adata[adata.obs['disease_global_check'].isin(['Immune disease','Infectious disease'])]
adata_can = adata[adata.obs['disease_global_check'] == 'Cancer']
adata_Neu = adata[adata.obs['disease_global_check'] == 'Neuronal disease']
adata_rare = adata[adata.obs['disease_global_check'].isin(['Cancer,Rare disease','Rare disease'])]


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict, KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import RobustScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def rf_model_cv(adata, target_cell_types, use_loocv=False, random_state=42):
    # 构建细胞类型比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)

    # 过滤掉细胞类型太少的个体
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少包含20种细胞类型
    df = df[zero_counts < max_zeros]

    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0
    X = df[target_cell_types]
    y = df['Age']

    # 标准化
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    # 参数搜索空间
    param_dist = {
        'n_estimators': [300],
        'max_depth': [None,30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    }

    rf = RandomForestRegressor(random_state=random_state)
    cv = LeaveOneOut() if use_loocv else KFold(n_splits=5, shuffle=True, random_state=random_state)

    rf_cv = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=50,
                               scoring='r2', cv=cv, random_state=random_state, n_jobs=-1, verbose=0)
    rf_cv.fit(X_scaled, y)

    best_rf = rf_cv.best_estimator_
    best_params = rf_cv.best_params_

    # 交叉验证预测
    y_cv_pred = cross_val_predict(best_rf, X_scaled, y, cv=cv)
    mse = mean_squared_error(y, y_cv_pred)
    r2 = r2_score(y, y_cv_pred)

    # 拟合最终模型
    best_rf.fit(X_scaled, y)
    y_pred = best_rf.predict(X_scaled)

    print("Random Forest 最佳参数：", best_params)
    print(f"Random Forest CV MSE: {mse:.4f}")
    print(f"Random Forest CV R²: {r2:.4f}")

    # 可视化：特征重要性
    importances = pd.Series(best_rf.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(10)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
    plt.title("Top 10 Feature Importances (Random Forest)")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    return best_rf, y, y_pred, mse, r2, scaler
    

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

def plot_rf_prediction(y_true, y_pred, extra_points=None, extra_labels=None):
    """
    可视化 Random Forest 模型的预测效果
    :param y_true: 实际年龄（Series 或 array）
    :param y_pred: 预测年龄（array）
    :param extra_points: 可选，额外点的坐标 [(x1, y1), (x2, y2), ...]
    :param extra_labels: 可选，额外点的标签 [label1, label2, ...]
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    # 评价指标
    mae = mean_absolute_error(y_true, y_pred)
    mad = np.median(np.abs(y_true - y_pred))
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pcc, _ = pearsonr(y_true, y_pred)

    # 回归线
    reg = LinearRegression().fit(y_true.reshape(-1, 1), y_pred)
    slope, intercept = reg.coef_[0], reg.intercept_

    # 绘图
    plt.figure(figsize=(8, 6))
    x_min, x_max = -10, 130
    plt.xlim(x_min, x_max)
    plt.ylim(x_min, x_max)
    plt.scatter(y_true, y_pred, c='dodgerblue', alpha=0.7, edgecolor='k', label='Samples')
    plt.plot([x_min, x_max], [x_min, x_max], 'r--', lw=2, label='Ideal line (y = x)')

    # 回归线
    x_line = np.linspace(x_min, x_max, 100)
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, 'g-', lw=2, label='Regression line')

    # 评价指标框
    plt.text(
        0.05, 0.95,
        f"$R^2$: {r2:.2f}\nMAE: {mae:.2f}\nMAD:{mad:.2f}\nPCC: {pcc:.2f}",
        transform=plt.gca().transAxes,
        fontsize=16,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )

    # 添加额外点
    if extra_points:
        for i, (x, y) in enumerate(extra_points):
            label = extra_labels[i] if extra_labels else 'Extra Point'
            plt.scatter(x, y, color='red', edgecolor='black', s=100, marker='o', label=label)

    # 去重图例
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='lower right')

    plt.xlabel('True Age')
    plt.ylabel('Predicted Age')
    plt.title('Random Forest Model: Predicted vs True Age')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
len(target_cell_types)


In [ ]:
rf_best, y_h, y_pred_h, mse, r2, scaler = rf_model_cv(adata_h, target_cell_types)


In [ ]:
plot_rf_prediction(y_h, y_pred_h, extra_points=None, extra_labels=None)



In [ ]:
##################################### 画 RF 的 age difference figure          ######################
def rf_age_diff(adata, rf_model, target_cell_types, scaler):
    import pandas as pd
    import numpy as np
    # 细胞比例计算
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少保留20种细胞类型
    df = df[zero_counts < max_zeros]
    print(f"剩余样本数: {len(df)}")
    # 保证目标细胞类型列存在
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0

    X = df[target_cell_types]
    y = df['Age']
    # 标准化
    X_scaled = scaler.transform(X)

    # 使用训练好的随机森林模型进行预测
    y_pred = rf_model.predict(X_scaled)
    y_pred = pd.Series(y_pred.flatten(), index=y.index)
    # 差值
    age_diff = y_pred - y
    return y, y_pred, age_diff


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_age_diff_boxplot(age_diff_dict):
    """
    绘制多组 age_diff 的箱线图，并使用 Mann-Whitney U 检验与第一组进行显著性标注。
    在计算和绘图前，去除每组数据中的异常值。

    参数:
    - age_diff_dict: 一个字典，key 为组名，value 为 age_diff 的 Series 或 array
    """
    def get_p_label(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'ns'
            
    def remove_outliers(data):
        """
        使用 IQR 方法去除数据中的异常值。
        处理输入为 pandas.Series 或 numpy.ndarray 的情况。
        """
        if isinstance(data, pd.Series):
            data = data.values  # 转换为 numpy 数组，方便进行 IQR 操作
        data = np.array(data)  # 确保 data 是 numpy 数组
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 返回去除异常值后的数据
        return data[(data >= lower_bound) & (data <= upper_bound)]

    # 整理数据为 DataFrame，并去除异常值
    plot_data = []
    for group, values in age_diff_dict.items():
        flat_vals = np.array(values).flatten()
        cleaned_vals = remove_outliers(flat_vals)  # 去除异常值
        plot_data.append(pd.DataFrame({'Age Difference': cleaned_vals, 'Group': group}))
    df_plot = pd.concat(plot_data, ignore_index=True)

    # 进行显著性检验：全部与第一个组（基准组）进行比较
    base_group_name = list(age_diff_dict.keys())[0]
    base_vals = np.array(remove_outliers(age_diff_dict[base_group_name])).flatten()  # 去除基准组异常值

    p_labels = []
    group_names = list(age_diff_dict.keys())
    for name in group_names[1:]:
        test_vals = np.array(remove_outliers(age_diff_dict[name])).flatten()  # 去除测试组异常值
        p = mannwhitneyu(base_vals, test_vals, alternative='two-sided').pvalue
        p_labels.append(get_p_label(p))

    # 绘图
    plt.figure(figsize=(0.9 * len(group_names), 5))
    ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2', showfliers=False)
    sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)

    y_max = df_plot['Age Difference'].max()
    y_min = df_plot['Age Difference'].min()
    y_range = y_max - y_min

    # 添加显著性标注
    def draw_sig_line(x1, x2, y, h, label):
        plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
        plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)

    for i, label in enumerate(p_labels):
        draw_sig_line(0, i+1, y_max + (i+1)*0.06*y_range, 0.02*y_range, label)
        
    plt.xlabel("Disease Type", fontsize=14)
    plt.ylabel("Age Difference", fontsize=14)
    plt.title('Predicted - True Age Difference(RF)', fontsize= 15)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

In [ ]:
y_h, y_peed_h, age_diff_h = rf_age_diff(adata_h, rf_best, target_cell_types, scaler)
y_immune, y_pred_immune, age_diff_immune = rf_age_diff(adata_immune, rf_best, target_cell_types, scaler)
y_can, y_pred_can, age_diff_can = rf_age_diff(adata_can, rf_best, target_cell_types, scaler)
y_rare, y_pred_rare, age_diff_rare = rf_age_diff(adata_rare, rf_best, target_cell_types, scaler)
y_neu, y_pred_neu, age_diff_neu = rf_age_diff(adata_Neu, rf_best, target_cell_types, scaler)


In [ ]:
age_diff_dict = {
        'Healthy': age_diff_h,
    'Immune': age_diff_immune,
    'Cancer': age_diff_can,
        'Rare': age_diff_rare,
        'Neuron': age_diff_neu
    }
plot_age_diff_boxplot(age_diff_dict)


In [ ]:
###################################### RF 模型减少特征试试效果有没有更好   #######################################
# target_cell_types_sub = ['','','','','','']


In [ ]:
################### Next try the Elastic Net model   ###################
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def en_model_cv(adata, target_cell_types, use_loocv=False):
    # 构建细胞类型比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)
    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31   # 最少有20种细胞类型
    df = df[zero_counts < max_zeros]
    remaining_samples = df.index.str[:-2].unique()
    total_cells = cell_counts.loc[remaining_samples].sum().sum()
    print(f"剩余个体数: {df.shape[0]}")
    print(f"剩余细胞总数: {total_cells}")

    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0
    X = df[target_cell_types]
    y = df['Age']

    # 标准化
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    # 交叉验证方式
    cv = LeaveOneOut() if use_loocv else KFold(n_splits=5, shuffle=True, random_state=42)

    # 构建ElasticNet模型（自动交叉验证选择alpha和l1_ratio）
    en_cv = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], 
                         alphas=np.logspace(-4, 2, 100), 
                         cv=cv, max_iter=10000)

    y_cv_pred = cross_val_predict(en_cv, X_scaled, y, cv=cv)
    mse = mean_squared_error(y, y_cv_pred)
    r2 = r2_score(y, y_cv_pred)

    # 拟合最终模型
    en_cv.fit(X_scaled, y)
    y_pred = en_cv.predict(X_scaled)

    print(f"ElasticNet 最佳 alpha: {en_cv.alpha_}")
    print(f"ElasticNet 最佳 l1_ratio: {en_cv.l1_ratio_}")
    print(f"ElasticNet CV MSE: {mse:.4f}")
    print(f"ElasticNet CV R²: {r2:.4f}")

    # 提取权重
    coef = pd.Series(en_cv.coef_, index=X.columns)
    top_features = coef.abs().sort_values(ascending=False).head(10)

    # 可视化：Top 权重细胞类型
    plt.figure(figsize=(8, 4))
    sns.barplot(x=top_features.values, y=top_features.index, palette='coolwarm')
    plt.title('Top 10 Cell Types (by Absolute Weight) in ElasticNet')
    plt.xlabel("Coefficient Weight")
    plt.tight_layout()
    plt.show()

    return en_cv, y, y_pred, mse, r2, scaler


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

def plot_en_prediction(y_true, y_pred, extra_points=None, extra_labels=None):
    """
    可视化 ElasticNet 模型的预测效果
    :param y_true: 实际年龄（Series 或 array）
    :param y_pred: 预测年龄（array）
    :param extra_points: 可选，额外点的坐标 [(x1, y1), (x2, y2), ...]
    :param extra_labels: 可选，额外点的标签 [label1, label2, ...]
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    # 评价指标
    mad = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pcc, _ = pearsonr(y_true, y_pred)

    # 回归线
    reg = LinearRegression().fit(y_pred.reshape(-1, 1), y_true)
    slope, intercept = reg.coef_[0], reg.intercept_

    # 绘图
    plt.figure(figsize=(8, 6))
    x_min, x_max = -10, 130
    plt.xlim(x_min, x_max)
    plt.ylim(x_min, x_max)
    plt.scatter(y_true, y_pred, c='dodgerblue', alpha=0.7, edgecolor='k', label='Samples')
    plt.plot([x_min, x_max], [x_min, x_max], 'r--', lw=2, label='Ideal line (y = x)')

    # 回归线
    x_line = np.linspace(x_min, x_max, 100)
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, 'g-', lw=2, label='ElasticNet regression line')

    # 评价指标框
    plt.text(
        0.05, 0.95,
        f"$R^2$: {r2:.2f}\nMAD: {mad:.2f}\nPCC: {pcc:.2f}",
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )

    # 添加额外点
    if extra_points:
        for i, (x, y) in enumerate(extra_points):
            label = extra_labels[i] if extra_labels else 'Extra Point'
            plt.scatter(x, y, color='red', edgecolor='black', s=100, marker='o', label=label)

    # 去重图例
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='lower right')

    plt.xlabel('True Age')
    plt.ylabel('Predicted Age')
    plt.title('ElasticNet Model: Predicted vs True Age')
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [ ]:
def en_age_diff(adata, en_model, target_cell_types, scaler):
    import pandas as pd
    import numpy as np

    # 细胞比例计算
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0).dropna(axis=0, how='any')

    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    df['sample'] = df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    df.set_index('sample', inplace=True)

    cell_prop_data = df.iloc[:, 2:]
    zero_counts = (cell_prop_data == 0).sum(axis=1)
    max_zeros = 31  # 至少保留20种细胞类型
    df = df[zero_counts < max_zeros]

    print(f"剩余样本数: {len(df)}")

    # 保证目标细胞类型列存在
    for ct in target_cell_types:
        if ct not in df.columns:
            df[ct] = 0

    X = df[target_cell_types]
    y = df['Age']

    # 标准化
    X_scaled = scaler.transform(X)

    # 预测
    y_pred = en_model.predict(X_scaled)
    y_pred = pd.Series(y_pred.flatten(), index=y.index)

    # 差值
    age_diff = y_pred - y

    return y, y_pred, age_diff


In [ ]:
en_best, y_h, y_pred_h, best_n, mse_list, scaler = en_model_cv(adata_h, target_cell_types)


In [ ]:
plot_en_prediction(y_h, y_pred_h, extra_points=None, extra_labels=None)


In [ ]:
disease_counts

In [ ]:
len(adata_h.obs['Celltype'].unique()), len(adata_immune.obs['Celltype'].unique()),len(adata_can.obs['Celltype'].unique()),len(adata_Neu.obs['Celltype'].unique()),len(adata_rare.obs['Celltype'].unique())


In [ ]:
y_h, y_peed_h, age_diff_h = en_age_diff(adata_h, en_best, target_cell_types, scaler)
y_immune, y_pred_immune, age_diff_immune = en_age_diff(adata_immune, en_best, target_cell_types, scaler)
y_can, y_pred_can, age_diff_can = en_age_diff(adata_can, en_best, target_cell_types, scaler)
y_Neu, y_pred_Neu, age_diff_Neu = en_age_diff(adata_Neu, en_best, target_cell_types, scaler)
y_rare, y_pred_rare, age_diff_rare = en_age_diff(adata_rare, en_best, target_cell_types, scaler)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_age_diff_boxplot(age_diff_dict):
    """
    绘制多组 age_diff 的箱线图，并使用 Mann-Whitney U 检验与第一组进行显著性标注。
    在计算和绘图前，去除每组数据中的异常值。

    参数:
    - age_diff_dict: 一个字典，key 为组名，value 为 age_diff 的 Series 或 array
    """

    def get_p_label(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'ns'

    def remove_outliers(data):
        """
        使用 IQR 方法去除数据中的异常值。
        处理输入为 pandas.Series 或 numpy.ndarray 的情况。
        """
        if isinstance(data, pd.Series):
            data = data.values  # 转换为 numpy 数组，方便进行 IQR 操作
        data = np.array(data)  # 确保 data 是 numpy 数组
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 返回去除异常值后的数据
        return data[(data >= lower_bound) & (data <= upper_bound)]

    # 整理数据为 DataFrame，并去除异常值
    plot_data = []
    for group, values in age_diff_dict.items():
        flat_vals = np.array(values).flatten()
        cleaned_vals = remove_outliers(flat_vals)  # 去除异常值
        plot_data.append(pd.DataFrame({'Age Difference': cleaned_vals, 'Group': group}))
    df_plot = pd.concat(plot_data, ignore_index=True)

    # 进行显著性检验：全部与第一个组（基准组）进行比较
    base_group_name = list(age_diff_dict.keys())[0]
    base_vals = np.array(remove_outliers(age_diff_dict[base_group_name])).flatten()  # 去除基准组异常值

    p_labels = []
    group_names = list(age_diff_dict.keys())
    for name in group_names[1:]:
        test_vals = np.array(remove_outliers(age_diff_dict[name])).flatten()  # 去除测试组异常值
        p = mannwhitneyu(base_vals, test_vals, alternative='two-sided').pvalue
        p_labels.append(get_p_label(p))

    # 绘图
    plt.figure(figsize=( 0.9* len(group_names), 5))
    ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2', showfliers=False)
    sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)

    y_max = df_plot['Age Difference'].max()
    y_min = df_plot['Age Difference'].min()
    y_range = y_max - y_min

    # 添加显著性标注
    def draw_sig_line(x1, x2, y, h, label):
        plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
        plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)

    for i, label in enumerate(p_labels):
        draw_sig_line(0, i+1, y_max + (i+1)*0.06*y_range, 0.02*y_range, label)

    plt.xlabel("Disease Type", fontsize=14)
    plt.ylabel("Age Difference", fontsize=14)
    plt.title('Predicted - True Age Difference(EN)', fontsize= 15)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

In [ ]:
age_diff_dict = {
        'Healthy':age_diff_h,
        'Immune':age_diff_immune,
        'Cancer':age_diff_can ,    'Neuron':age_diff_Neu,
    'Rare':age_diff_rare
    }

plot_age_diff_boxplot(age_diff_dict)

In [ ]:
# len(adata_h.obs['sample'].unique())


In [ ]:
################ 再试一下 PLSR 方法   ################ 
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score

def plsr_model(adata, target_cell_types, n_components):
    # Step 1: 构建每个样本中各个细胞类型的比例表
    cell_counts = adata.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
    cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
    cell_props = cell_props.dropna(axis=0, how='any')
    # 重置索引
    df = cell_props.reset_index()
    df['sample'] = df['sample'].astype(str)
    # 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
    df['sample'] = (
        df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
    )
    df.set_index('sample', inplace=True)
    # Step 2: 提取感兴趣的细胞类型的比例作为X
    X = df[target_cell_types]

    # Step 3: 构建y（年龄）
    y = df['Age']
    
    # Step 4: 标准化特征
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Step 5: 构建PLSR模型（设定成1-3个主成分可以交叉验证选择）
    pls = PLSRegression(n_components=n_components)
    pls.fit(X_scaled, y)
    y_pred = pls.predict(X_scaled)
    return pls, y, y_pred



In [ ]:
# len(adata_h.obs['sample'].unique())
# adata_h = adata[(adata.obs['disease_global_check'].isin(['Normal', 'Healthy'])) & (adata.obs['Age']!=0)]


In [ ]:
target_cell_types = list(set(adata_h.obs['Celltype']))


In [ ]:
pls, y_h, y_pred_h = plsr_model(adata_h, target_cell_types, n_components=10)


import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import numpy as np

# 计算评价指标
mad = mean_absolute_error(y_h, y_pred_h)
mse = mean_squared_error(y_h, y_pred_h)
rmse = np.sqrt(mse)
r2 = r2_score(y_h, y_pred_h)
pcc, _ = pearsonr(y_h, y_pred_h)

# 打印指标
print(f"MAD（Mean Absolute Deviation）: {mad:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² score: {r2:.4f}")
print(f"PCC (Pearson Correlation Coefficient): {pcc:.4f}")

# 绘图
plt.figure(figsize=(6, 6))
plt.scatter(y_h, y_pred_h, alpha=0.8, edgecolor='k', s=60, c='skyblue', label='Samples')
plt.plot([y_h.min(), y_h.max()], [y_h.min(), y_h.max()], 'r--', lw=2, label='Ideal Fit')

# 美化图像
plt.xlabel("Chronological Age", fontsize=12)
plt.ylabel("Predicted Age", fontsize=12)
plt.title("PLSR Age Prediction", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# 添加文本：R²、MAD 和 PCC
plt.text(
    0.05, 0.95,
    f"$R^2$: {r2:.2f}\nMAD: {mad:.2f}\nPCC: {pcc:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
)

plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Cancer 

cell_counts = adata_can.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_can = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_can = pls.predict(X_scaled) # 
age_diff_can = y_pred_can -  y_can
age_diff_can


In [ ]:
# cell_counts


In [ ]:
# Immune

cell_counts = adata_immune.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_immune = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_immune = pls.predict(X_scaled) # 
age_diff_immune = y_pred_immune -  y_immune
age_diff_immune


In [ ]:
# Neuron

cell_counts = adata_Neu.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_neu = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_neu = pls.predict(X_scaled) # 
age_diff_neu = y_pred_neu -  y_neu
age_diff_neu


In [ ]:
len(target_cell_types)


In [ ]:
# Rare

cell_counts = adata_rare.obs.groupby(['sample', 'Age', 'Celltype']).size().unstack(fill_value=0)
cell_props = cell_counts.div(cell_counts.sum(axis=1), axis=0)
cell_props = cell_props.dropna(axis=0, how='any')
# 重置索引
df = cell_props.reset_index()
df['sample'] = df['sample'].astype(str)
# 给同一个 sample 下的每个年龄编号（A01_0、A01_1 等）
df['sample'] = (
    df.groupby('sample').cumcount().astype(str).radd(df['sample'] + '_')
)
df.set_index('sample', inplace=True)
# Step 2: 提取感兴趣的细胞类型的比例作为X
for ct in target_cell_types:
    if ct not in df.columns:
        df[ct] = 0  # 不存在的细胞类型比例设为0

# 保证列顺序和 target_cell_types 一致
X = df[target_cell_types]
# Step 3: 构建y（年龄）
y_rare = df['Age'] 
# Step 4: 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y_pred_rare = pls.predict(X_scaled) # 
age_diff_rare = y_pred_rare -  y_rare
age_diff_rare


In [ ]:
############################### PLSR 的 age difference
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# 转为一维数组
healthy_vals = age_diff_h.to_numpy().flatten()
can_vals = age_diff_can.to_numpy().flatten()
immune_vals = age_diff_immune.to_numpy().flatten()
neu_vals = age_diff_neu.to_numpy().flatten()
rare_vals = age_diff_rare.to_numpy().flatten()

# 整理绘图数据
df_plot = pd.DataFrame({
    'Age Difference': np.concatenate([healthy_vals, can_vals,immune_vals, neu_vals,rare_vals]),
    'Group': (['Healthy'] * len(healthy_vals) +
              ['Cancer'] * len(can_vals) +
              ['Immune'] * len(immune_vals) +
              ['Neuron'] * len(neu_vals) + ['Rare'] * len(rare_vals)) 
})

# Mann-Whitney U检验
def get_p_label(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

# 只和 Healthy 组比
p_hc = mannwhitneyu(healthy_vals, can_vals, alternative='two-sided').pvalue
p_hi = mannwhitneyu(healthy_vals, immune_vals, alternative='two-sided').pvalue
p_hn = mannwhitneyu(healthy_vals, neu_vals, alternative='two-sided').pvalue
p_hr = mannwhitneyu(healthy_vals, rare_vals, alternative='two-sided').pvalue

label_hc = get_p_label(p_hc)
label_hi = get_p_label(p_hi)
label_hn = get_p_label(p_hn)
label_hr = get_p_label(p_hr)

# 可视化
plt.figure(figsize=(5, 6))
ax = sns.boxplot(data=df_plot, x='Group', y='Age Difference', palette='Set2')
sns.stripplot(data=df_plot, x='Group', y='Age Difference', color='black', size=4, alpha=0.5)

# 获取最大 y 值用于标注位置
y_max = df_plot['Age Difference'].max()
y_min = df_plot['Age Difference'].min()
y_range = y_max - y_min

# 添加显著性连线和星号
def draw_sig_line(x1, x2, y, h, label):
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)/2, y+h+0.01*y_range, label, ha='center', va='bottom', fontsize=12)

draw_sig_line(0, 1, y_max + 0.02*y_range, 0.02*y_range, label_hc)  # Healthy vs COVID
draw_sig_line(0, 2, y_max + 0.08*y_range, 0.02*y_range, label_hi)  # Healthy vs AML
draw_sig_line(0, 3, y_max + 0.14*y_range, 0.02*y_range, label_hn)  # Healthy vs IAV
draw_sig_line(0, 4, y_max + 0.2*y_range, 0.02*y_range, label_hr)  # Healthy vs IAV

plt.xlabel("Disease Type", fontsize=14)
plt.ylabel("Age Difference", fontsize=14)
plt.title('Predicted - True Age Difference(PLSR)', fontsize= 15)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:


############### 改成 4 个subtype后，产王老师说的那种 boxplot, 柱子窄一点的 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
# 设置Nature/Cell期刊风格的参数
plt.style.use('default')
rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})
# 1. 提取数据框并添加Celltype信息
adata_meta = adata.obs[['leiden','Celltype_general','Celltype','score_Tip', 'score_antigen_pre', 'score_stalk']].copy()
rename_dict = {
    'score_Tip': 'Tip',
    'score_antigen_pre': 'AP',
    'score_stalk': 'Stalk'
}
adata_meta = adata_meta.rename(columns=rename_dict)
adata_meta['Celltype'] = adata_meta['Celltype'].astype(str) + '_C' + adata_meta['leiden'].astype(str)
# 定义leiden到Celltype的映射字典

# 确保leiden列是字符串类型，方便处理
adata_meta['leiden'] = adata_meta['leiden'].astype(str)
# 3. 按照leiden分组计算平均值
# 注意：这里修正了列名大小写问题
feature_columns = ['Tip', 'AP', 'Stalk']
# 检查列名是否存在，确保大小写匹配
available_columns = [col for col in feature_columns if col in adata_meta.columns]
grouped_mean = adata_meta.groupby(['Celltype'])[available_columns].mean().reset_index()
grouped_mean['leiden'] = grouped_mean['Celltype'].str.split('_C').str[-1].astype(int)
grouped_mean = grouped_mean.sort_values(['Celltype','leiden'])
print(grouped_mean[['leiden', 'Celltype']].to_string(index=False))
# 5. 可视化设置
n_leiden = len(grouped_mean)
print(f"\nLeiden群数: {n_leiden}")
# 设置子图布局：4行10列
n_rows = 3
n_cols = 11
n_subplots = n_rows * n_cols
# 如果leiden数量少于子图数量，调整布局
if n_leiden < n_subplots:
    n_cols = int(np.ceil(n_leiden / n_rows))
    n_subplots = n_rows * n_cols

colors = plt.cm.Set2(np.linspace(0, 1, len(available_columns)))
# 6. 创建图形
fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.5*n_cols, 1.6*n_rows))
axes = axes.flatten()
# 设置图形标题
fig.suptitle('Signature score of endo molecular markers',  # Signature score of subFibro(long list)
             fontsize=12, fontweight='bold', y=1.02)

# 7. 绘制每个leiden的柱状图（添加数值标签）
for i, row in enumerate(grouped_mean.itertuples()):
    if i >= len(axes):
        break
    
    leiden = row.leiden
    celltype = row.Celltype
    ax = axes[i]
    
    # 提取当前leiden的数据
    leiden_data = grouped_mean.loc[grouped_mean['leiden'] == leiden, available_columns].iloc[0]
    
    # 创建柱状图
    x_pos = np.arange(len(available_columns))
    bars = ax.bar(x_pos, leiden_data.values, color=colors, edgecolor='black', linewidth=0.5)
    
    # 添加数值标签到每个柱子顶部
    for bar_idx, (bar, value) in enumerate(zip(bars, leiden_data.values)):
        # 确定标签位置
        height = bar.get_height()
        
        # 如果值太小，将标签放在柱子内部顶部
        if height < grouped_mean[available_columns].values.max() * 0.05:
            # 小值：标签放在柱子内部，白色字体
            text_y = height * 0.8  # 放在柱子高度的80%位置
            color = 'black' # 小的值也还是 black
            va = 'bottom'
        else:
            # 正常值：标签放在柱子顶部上方
            text_y = height + grouped_mean[available_columns].values.max() * 0.01  # 稍微高于柱子
            color = 'black'
            va = 'bottom'
        
        # 格式化数值显示（根据数值大小选择格式）
        if abs(value) < 0.001:
            text = f'{value:.1e}'
        elif abs(value) < 0.01:
            text = f'{value:.4f}'
        elif abs(value) < 0.1:
            text = f'{value:.3f}'
        else:
            text = f'{value:.2f}'
        
        # 添加文本标签
        ax.text(
            bar_idx, text_y, text,
            ha='center', va=va,
            fontsize=6,  # 使用较小字体
            color=color,
            fontweight='bold' if color == 'white' else 'normal',
            rotation=0
        )
    
    # 设置子图标题（包含leiden和Celltype信息）
    # 缩短Celltype显示，只显示主要部分
    if len(celltype) > 20:
        # 如果Celltype太长，只显示前部分
        short_celltype = celltype # celltype[:17] + "..."
    else:
        short_celltype = celltype
    
    title_text =f'{short_celltype}'  # f'L{leiden}\n{short_celltype}'
    ax.set_title(title_text, fontsize=6, fontweight='bold', pad=3)
    
    # 隐藏x轴刻度
    ax.set_xticks([])
    
    # 设置y轴标签
    if i % n_cols == 0:  # 每行最左边的图显示y轴标签
        ax.set_ylabel('Signature Score', fontsize=8)
    
    # 设置y轴范围，为顶部标签留出空间
    y_max = grouped_mean[available_columns].values.max() * 1.15  # 增加15%的空间用于标签
    ax.set_ylim([0, y_max])
    
    # 添加网格线（仅y方向，更简洁）
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # 移除顶部和右侧边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 根据Celltype前缀添加背景色区分
    celltype_prefix =  celltype.split('_')[0]     # get_celltype_prefix(celltype) , 这个函数不在了
    if celltype_prefix == 'ArtEC':  # iFibro
        ax.set_facecolor('#FFF5E6')  # 浅橙色背景
    elif celltype_prefix == 'CapEC':  # eFibro
        ax.set_facecolor('#E6F7FF')  # 浅蓝色背景
    elif celltype_prefix == 'EndoMT-II':  # Myfibro
        ax.set_facecolor('#F0E6FF')  # 浅紫色背景
    elif celltype_prefix == 'EndoMT-I':  # qFibro
        ax.set_facecolor('#E6FFE6')  # 浅绿色背景
    elif celltype_prefix == 'LymEC':  # eFibro
        ax.set_facecolor("#E58606")  
    elif celltype_prefix == 'PC':  # Myfibro
        ax.set_facecolor("#BEBADA")  
    elif celltype_prefix == 'SMC':  # qFibro
        ax.set_facecolor("#9C9C5E")  
    elif celltype_prefix == 'VenEC':  # qFibro
        ax.set_facecolor("#F3D4F4")  # 

# 8. 隐藏多余的子图
for i in range(len(grouped_mean), len(axes)):
    axes[i].axis('off')

# 9. 创建统一的图例
# 使用简化版特征名称作为图例标签
feature_short_names = []
for col in available_columns:
    if 'Tip' in col:
        feature_short_names.append('Tip')
        # feature_short_names.append('Myfibro_l')
    elif 'AP' in col:
        feature_short_names.append('AP')
    elif 'Stalk' in col:
        feature_short_names.append('Stalk')

# 创建自定义的图例句柄
legend_handles = [plt.Rectangle((0,0),1,1, color=colors[j], edgecolor='black', linewidth=0.5) 
                  for j in range(len(available_columns))]


# 创建Celltype分组图例
celltype_prefix_colors = {
    1: ('ArtEC', '#FFF5E6'),
    2: ('CapEC', '#E6F7FF'),
    3: ('EndoMT-II', '#F0E6FF'),
    4: ('EndoMT-I', '#E6FFE6'),
        5: ('LymEC', '#E58606'),
    6: ('PC', '#BEBADA'),
    7: ('SMC', '#9C9C5E'),
    8: ('VenEC', '#F3D4F4')
}

# 创建Celltype图例句柄
celltype_handles = []
celltype_labels = []

for prefix in sorted(celltype_prefix_colors.keys()):
    label, color = celltype_prefix_colors[prefix]
    handle = plt.Rectangle((0,0),1,1, color=color, edgecolor='gray', linewidth=0.5, alpha=0.7)
    celltype_handles.append(handle)
    celltype_labels.append(label)

# Subtypes图例（左下区域）
fig.legend(legend_handles, feature_short_names, 
           loc='lower center', 
           bbox_to_anchor=(0.2, 0.01),  # 底部左侧
           ncol=len(available_columns),  # 水平排列
           fontsize=8,
           title='marker_list',
           title_fontsize=9,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

# Celltype Groups图例（右下区域）
fig.legend(celltype_handles, celltype_labels,
           loc='lower center',
           bbox_to_anchor=(0.6, 0.01),  # 底部右侧
           ncol=8,  # 水平排列
           title='Celltype',
           title_fontsize=9,
           fontsize=8,
           frameon=True,
           fancybox=True,
           shadow=False,
           framealpha=0.9)

plt.tight_layout()
plt.subplots_adjust(top=0.95, bottom=0.11,left=0.05, right=0.9, hspace=0.4, wspace=0.3)

# 12. 保存图形
output_path = './code/Endo_leiden_signature_scores.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"图形已保存到: {output_path}")

# 13. 显示图形
plt.show()
